# Santander Product Recommendation

Recommending which banking products a customer will acquire next month, given their current portfolio, demographics, and history.

## The question

The competition asks a specific question: *"predict what a customer will buy in addition to what they already had at 2016-05-28"*. Not what they hold — what they **acquire**. That distinction drives every decision in this notebook, from the type inference to the choice of loss function.

## Results

| Version | Approach | Private Score |
|---|---|---|
| Naive popularity | Top-7 historical additions, masked | 0.01813 |
| Collaborative only | LightFM matrix factorization, no features | 0.01945 |
| v1 | LightFM + simple demographics | 0.02269 |
| **v6 final** | **LightFM + ordinal encoding + rare labels, no weights** | **0.02491** |

The final pipeline improves on popularity by 36.6%. The path to get there was not linear some example failed because adding diagnostic features that were informative for *understanding* the dataset did not help *predicting* the target. The successful v6 keeps the model simple and focuses the feature engineering on regularization.

## Why the failure is documented

Notebooks that only show the final state hide the reasoning that produced it. The v5 failure (0.01407, worse than popularity) is presented alongside the successes because it teaches a lesson: feature engineering guided by dataset understanding rather than target alignment backfires. Anyone reading this notebook should learn from the failure, not just the win.

## Structure

1. Data loading and schema inference
2. Deep diagnostics: understanding the dataset before modeling
3. Target characterization: what the prediction problem actually looks like
4. Three models: naive popularity, pure collaborative, hybrid pipeline
5. Temporal validation: April → May holdout
6. Final submission: May → June
7. Lessons

## Setup

The dependencies are minimal. Polars handles data loading and transformation (fast, typed, lazy). LightFM handles the recommendation model. Everything else is standard scientific Python.

In [ ]:
!pip install lightfm-next
# !pip install scikit-surprise-2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 8.7 MB/s  0:00:00


In [ ]:

from loaders import (
   load_data_rigorous, load_data_rigorous_polars, rename_columns, to_snake_case,
   skim,drop_empty_and_constant,to_category,downcast_numerics,clean_dataframe
)

from pandas_transformers import (
    categorizar_todo_pandas,Flagnan3, Flagout2, ZeroInflatedTransformer,ZeroInflatedTransformerPandas2, PercentileRanker,PercentileRankerPandas2,
    RareLabelPandas, Oneh, TargetEncoderFast, TargetEncoderFastPandas2,SklearnPolynomialWrapper, Escaleo, Bini,BiniPandas2, HashingEncoderCustom
)
from polars_transformers import (
    categorizar_todo_polars,categorizar_todo_polars, Flagnan3Polars,FlagnanPolars, Flagout2Polars, ZeroInflatedTransformerPolars,BiniPolars2,
    PercentileRankerPolars, RareLabelPolars, OnehPolars, TargetEncoderFastPolars,CateEncodePolars, BiniPolars, EscaleoPolars,OrdinalEncPolars,DropConstantFeaturesPolars
)
from dashboards2 import (
    classify_variables,
    fast_eda_report,
    analyze_numerical_distributions,
    get_classification_feature_dashboard,
    get_regression_feature_dashboard,
    audit_clas_norm,
    audit_regre_norm,
    audit_fe_information_gain,
    plot_classification_correlations_plotly,
    plot_classification_correlations_plotly_r
)

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
from datetime import date
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from lightfm import LightFM
from lightfm.data import Dataset as LightFMDataset
import os
import warnings
warnings.filterwarnings("ignore")

from google.colab import userdata
kag=userdata.get('kaggle')

os.environ["KAGGLE_API_TOKEN"] =kag

pl.Config.set_tbl_rows(25)
pl.Config.set_tbl_cols(-1)

polars.config.Config

In [ ]:
!kaggle competitions download -c santander-product-recommendation
!unzip -o santander-product-recommendation.zip
!unzip -o train_ver2.csv.zip
!unzip -o test_ver2.csv.zip
!unzip -o sample_submission.csv.zip

100% 228M/228M [00:01<00:00, 120MB/s]

Archive:  santander-product-recommendation.zip
  inflating: sample_submission.csv.zip  
  inflating: test_ver2.csv.zip       
  inflating: train_ver2.csv.zip      
Archive:  train_ver2.csv.zip
  inflating: train_ver2.csv          
Archive:  test_ver2.csv.zip
  inflating: test_ver2.csv           
Archive:  sample_submission.csv.zip
  inflating: sample_submission.csv   


## Data loading

The raw CSV mixes numeric, categorical, date, and genuinely mixed columns. Reading with `infer_schema_length=0` forces every column to string, which guarantees the loader never fails mid-file on a stray label. The diagnostic that follows reconstructs the real types from the data.

This is the most important architectural decision in the pipeline. Every downstream failure in the iteration history traces back to a wrong type inference or an ID leak introduced by treating an identifier as a feature. Getting the schema right from the start prevents those problems.

Three files from the Kaggle competition:

- `train_ver2.csv` — 13.6M rows, 17 months from January 2015 to May 2016
- `test_ver2.csv` — 929,615 rows, one per customer to score for June 2016
- `sample_submission.csv` — the expected format (ncodpers, then space-separated product list)

In [ ]:
DATA_PATH = "/content/train_ver2.csv"
TEST_PATH = "/content/test_ver2.csv"
SAMPLE_SUB = "/content/sample_submission.csv"

NULL_LIKE = [
    "", " ", "NA", "N/A", "NaN", "nan", "null", "NULL",
    "None", "undefined", "NA\x00",
]

# Read every column as text so the scan never fails on type inference
raw_lf = pl.scan_csv(
    DATA_PATH,
    null_values=NULL_LIKE,
    infer_schema_length=0,
)

# 10% deterministic hash sample for profiling (not head, which would concentrate
# the sample in early months because the file is sorted by date)
sample_df = (
    raw_lf
    .filter(pl.col("ncodpers").hash() % 10 == 0)
    .collect()
)

print(f"Sample: {len(sample_df):,} rows, {len(sample_df.columns)} columns")
print(f"Months covered: {sample_df['fecha_dato'].n_unique()}")

Sample: 1,358,466 rows, 48 columns
Months covered: 17


In [ ]:
skim(sample_df.to_pandas())

── Data Summary ────────────────────────────────────────────────────────────────────────────────────
type       value
-------  -------
Rows     1358466
Columns       48

Column type frequency:
          Count
------  -------
object       48


── Variable type: object ───────────────────────────────────────────────────────────────────────────
    name                     na_count    na_count_fr    n_unique  top_counts
--  ---------------------  ----------  -------------  ----------  ---------------------------------------------------------------------------------------------
 0  fecha_dato                      0        0                17  2016-05-28: 92737 (6.8%), 2016-04-28: 92409 (6.8%), 2016-03-28: 92061 (6.8%)
 1  ncodpers                        0        0             95199  538029: 17 (0.0%),  538108: 17 (0.0%),  538010: 17 (0.0%)
 2  ind_empleado                 2563        0.00189           4  N: 1355053 (99.9%), B: 357 (0.0%), A: 255 (0.0%)
 3  pais_residencia              2563

## Schema diagnostic

The diagnostic reports facts about each column. It does not decide what to do with them. For every column it records missingness, cardinality, parseability as int and float, fractional content, non-numeric labels, and date-pattern match rate.

The classification into types happens *after*, as a rule applied to these facts. Keeping observation and decision separate prevents silent data loss. A column that is 88% numeric and 12% labels will show `non_numeric_pct = 12` rather than being quietly classified as numeric and losing its labels on cast.

Three columns in this dataset carry irregularities that a naive cast would destroy:

- `age` and `antiguedad` contain literal `"NA"` strings that arrive padded with whitespace and are not caught by the CSV `null_values`. These should be null, not zero.
- `indrel_1mes` mixes numeric values `1, 2, 3, 4` with their decimal variants and a small number of `P` labels representing potential customers (~874 rows in the full dataset).
- `canal_entrada` is genuinely mixed: 99.6% alphanumeric channel codes, 0.4% legacy numeric codes.

The first two encode automatically. The third stays as text.

In [ ]:

DATE_FORMATS = [
    (r"^\d{4}-\d{2}-\d{2}$", "%Y-%m-%d"),
    (r"^\d{4}/\d{2}/\d{2}$", "%Y/%m/%d"),
    (r"^\d{2}-\d{2}-\d{4}$", "%d-%m-%Y"),
    (r"^\d{2}/\d{2}/\d{4}$", "%d/%m/%Y"),
]

NULL_LIKE = {"NA", "N/A", "NAN", "NULL", "NONE", "UNDEFINED", "", "UNKNOWN"}

HIGH_NULL_THRESHOLD = 99.0
ZERO_VARIANCE_MAX = 1
DATE_MATCH_MIN = 95.0
TEXT_PURITY_MIN = 95.0
SMALL_LABELS_MAX = 10
def diagnose_polars(sample_df: pl.DataFrame) -> pl.DataFrame:
    total = len(sample_df)
    rows = []

    for col in sample_df.columns:
        s = sample_df[col]
        nulls = s.null_count()
        non_null = total - nulls

        if non_null == 0:
            rows.append({
                "column": col, "null_pct": 100.0, "n_unique": 0,
                "parses_int": False, "parses_float": False,
                "non_numeric_pct": 0.0, "n_dirty_distinct": 0,
                "date_match_pct": 0.0, "has_decimals": False,
                "date_format": None, "dirty_samples": [],
                "min": None, "max": None,
            })
            continue

        cleaned = s.drop_nulls().str.strip_chars()
        n_unique = cleaned.n_unique()

        as_float = cleaned.cast(pl.Float64, strict=False)
        num_vals = as_float.drop_nulls()
        n_num_ok = len(num_vals)

        has_decimals = False
        is_int_ok = False
        if n_num_ok > 0:
            has_decimals = bool((num_vals % 1 != 0).any())
            # parses_int only when every value parses and none carries a fractional part
            if n_num_ok == non_null:
                is_int_ok = not has_decimals

        n_non_num = non_null - n_num_ok
        dirty = cleaned.filter(as_float.is_null())
        n_dirty_distinct = dirty.n_unique()
        dirty_samples = dirty.unique().head(10).to_list()

        best_pct, best_fmt = 0.0, None
        for pattern, fmt in DATE_FORMATS:
            matched = cleaned.str.contains(pattern)
            pct = (matched.sum() / non_null) * 100
            if pct > best_pct:
                best_pct = pct
                best_fmt = fmt

        rows.append({
            "column": col,
            "null_pct": round(nulls / total * 100, 2),
            "n_unique": n_unique,
            "parses_int": is_int_ok,
            "parses_float": bool(n_num_ok == non_null),
            "non_numeric_pct": round(n_non_num / non_null * 100, 2),
            "n_dirty_distinct": n_dirty_distinct,
            "date_match_pct": round(best_pct, 2),
            "date_format": best_fmt if best_pct > 0 else None,
            "has_decimals": has_decimals,
            "dirty_samples": dirty_samples,
            "min": float(num_vals.min()) if len(num_vals) > 0 else None,
            "max": float(num_vals.max()) if len(num_vals) > 0 else None,
        })

    return pl.DataFrame(rows)


def pick_int_dtype_polars(lo, hi):
    if lo is None or hi is None:
        return pl.Int64
    if lo >= 0:
        if hi <= 200: return pl.UInt8
        if hi <= 60_000: return pl.UInt16
        if hi <= 4_000_000_000: return pl.UInt32
    if -100 <= lo and hi <= 100: return pl.Int8
    if -30_000 <= lo and hi <= 30_000: return pl.Int16
    if -2_000_000_000 <= lo and hi <= 2_000_000_000: return pl.Int32
    return pl.Int64




def classify_polars(row: dict) -> str:
    if row["null_pct"] >= HIGH_NULL_THRESHOLD:
        return "drop_high_null"
    if row["n_unique"] <= ZERO_VARIANCE_MAX:
        return "drop_constant"
    if row["date_match_pct"] >= DATE_MATCH_MIN:
        return "date"
    if row["parses_int"]:
        # 0/1 columns are flags; anything else is a plain integer
        if row["n_unique"] == 2 and row["min"] == 0 and row["max"] == 1:
            return "flag"
        return "int"
    if row["parses_float"]:
        return "float"
    if row["non_numeric_pct"] >= TEXT_PURITY_MIN:
        return "text"
    # Single non-numeric label that looks like a null placeholder
    if row["n_dirty_distinct"] == 1 and str(row["dirty_samples"][0]).strip().upper() in NULL_LIKE:
        return "numeric_with_null_label"
    # Mixed numeric + a few labels: safe to encode only when numeric part is integral
    if row["n_dirty_distinct"] <= SMALL_LABELS_MAX:
        if row["has_decimals"] or (row["max"] is not None and row["max"] > 1000):
            # Fractional or very large: encoding would collide with the label codes
            return "numeric_with_null_label"
        return "mixed_small_labels_int"
    return "text"


In [ ]:

def build_curation_plan_polars(diag: pl.DataFrame, sample_df: pl.DataFrame) -> dict:
    plan = {"drop": [], "transforms": {}, "mixed_labels": {}}

    for row in diag.iter_rows(named=True):
        kind = classify_polars(row)
        col = row["column"]

        if kind in ("drop_high_null", "drop_constant"):
            plan["drop"].append(col)

        elif kind == "date":
            plan["transforms"][col] = ("date", row["date_format"])

        elif kind == "flag":
            plan["transforms"][col] = ("flag", pl.UInt8)

        elif kind == "int":
            dtype = pick_int_dtype_polars(row["min"], row["max"])
            plan["transforms"][col] = ("int", dtype)

        elif kind == "float":
            plan["transforms"][col] = ("float", pl.Float64)

        elif kind == "numeric_with_null_label":
            # Preserve integer dtype when the numeric part has no decimals (e.g. age)
            if row["has_decimals"]:
                plan["transforms"][col] = ("numeric_with_null_label", pl.Float64)
            else:
                dtype = pick_int_dtype_polars(row["min"], row["max"])
                plan["transforms"][col] = ("numeric_with_null_label", dtype)

        elif kind == "text":
            plan["transforms"][col] = ("text", pl.String)

        elif kind == "mixed_small_labels_int":
            s_clean = sample_df[col].drop_nulls().str.strip_chars()
            as_float = s_clean.cast(pl.Float64, strict=False)
            labels = s_clean.filter(as_float.is_null()).unique().to_list()
            num_series = as_float.drop_nulls()
            max_num = float(num_series.max()) if len(num_series) > 0 else 0.0
            mapping = {lbl: int(max_num) + i + 1 for i, lbl in enumerate(labels)}
            plan["transforms"][col] = ("mixed_small_labels_int", mapping)
            plan["mixed_labels"][col] = mapping

    return plan


def print_plan(plan: dict) -> None:
    by_kind = {}
    for col, (kind, _meta) in plan["transforms"].items():
        by_kind.setdefault(kind, []).append(col)

    print(f"dropped ({len(plan['drop'])}): {plan['drop']}")
    for kind in ("date", "flag", "int", "float", "numeric_with_null_label",
                 "text", "mixed_small_labels_int"):
        cols = by_kind.get(kind, [])
        if cols:
            print(f"{kind:<25} ({len(cols)}): {cols}")
    for col, mapping in plan["mixed_labels"].items():
        print(f"mixed mapping {col}: {mapping}")

def apply_curation_polars(lf: pl.LazyFrame, plan: dict) -> pl.LazyFrame:
    lf = lf.drop(plan["drop"])
    exprs = []

    for col, (kind, meta) in plan["transforms"].items():
        if kind == "date":
            exprs.append(
                pl.col(col).str.strip_chars().str.to_date(meta, strict=False).alias(col)
            )

        elif kind == "flag":
            exprs.append(
                pl.col(col).str.strip_chars()
                .cast(pl.UInt8, strict=False)
                .fill_null(0)
                .alias(col)
            )

        elif kind == "int":
            exprs.append(
                pl.col(col).str.strip_chars().cast(meta, strict=False).alias(col)
            )

        elif kind == "float":
            exprs.append(
                pl.col(col).str.strip_chars().cast(pl.Float64, strict=False).alias(col)
            )

        elif kind == "numeric_with_null_label":
            stripped = pl.col(col).str.strip_chars()
            if meta == pl.Float64:
                exprs.append(stripped.cast(pl.Float64, strict=False).alias(col))
            else:
                # Strip trailing .0 first so "32.0" casts cleanly to an integer dtype
                exprs.append(
                    stripped.str.replace(r"\.0$", "")
                    .cast(meta, strict=False)
                    .alias(col)
                )

        elif kind == "text":
            exprs.append(
                pl.col(col).str.strip_chars().fill_null("UNKNOWN").alias(col)
            )

        elif kind == "mixed_small_labels_int":
            base = pl.col(col).str.strip_chars()
            as_num = base.cast(pl.Float64, strict=False)
            expr = as_num
            for lbl, code in meta.items():
                expr = pl.when(base == lbl).then(pl.lit(float(code))).otherwise(expr)
            exprs.append(expr.cast(pl.Int16, strict=False).alias(col))

    return lf.with_columns(exprs) if exprs else lf





diag_polars = diagnose_polars(sample_df)
plan_polars = build_curation_plan_polars(diag_polars, sample_df)
print_plan(plan_polars)

curated_lf = apply_curation_polars(raw_lf, plan_polars)
print()
print(curated_lf.collect_schema())

dropped (3): ['ult_fec_cli_1t', 'conyuemp', 'tipodom']
date                      (2): ['fecha_dato', 'fecha_alta']
flag                      (26): ['ind_nuevo', 'ind_actividad_cliente', 'ind_ahor_fin_ult1', 'ind_aval_fin_ult1', 'ind_cco_fin_ult1', 'ind_cder_fin_ult1', 'ind_cno_fin_ult1', 'ind_ctju_fin_ult1', 'ind_ctma_fin_ult1', 'ind_ctop_fin_ult1', 'ind_ctpp_fin_ult1', 'ind_deco_fin_ult1', 'ind_deme_fin_ult1', 'ind_dela_fin_ult1', 'ind_ecue_fin_ult1', 'ind_fond_fin_ult1', 'ind_hip_fin_ult1', 'ind_plan_fin_ult1', 'ind_pres_fin_ult1', 'ind_reca_fin_ult1', 'ind_tjcr_fin_ult1', 'ind_valo_fin_ult1', 'ind_viv_fin_ult1', 'ind_nomina_ult1', 'ind_nom_pens_ult1', 'ind_recibo_ult1']
int                       (3): ['ncodpers', 'indrel', 'cod_prov']
float                     (1): ['renta']
numeric_with_null_label   (2): ['age', 'antiguedad']
text                      (10): ['ind_empleado', 'pais_residencia', 'sexo', 'tiprel_1mes', 'indresi', 'indext', 'canal_entrada', 'indfall', 'nomprov', 'segmen

## Curation

Rules applied to the diagnostic facts. Each column maps to one strategy:

- `drop_high_null` — over 99% missing, not worth imputing
- `drop_constant` — single unique value, no information
- `date` — 95%+ match a known date pattern
- `flag` — integers restricted to `{0, 1}`
- `int` / `float` — parse cleanly as their respective types
- `text` — over 95% non-numeric
- `numeric_with_null_label` — one non-numeric label that looks like a null placeholder
- `mixed_small_labels_int` — mostly integers plus a few labels; encoded as `max + 1, max + 2, ...` when the numeric part is integral

The last case recovers `P` in `indrel_1mes`. The mapping is derived from the data, not hardcoded: `P` becomes `max_num + 1`, which is 5. That is the principle throughout — rules derived from the data, not assumptions about the data.

## Deep target characterization

Before modeling, we characterize the prediction problem across four dimensions. Each number below shapes a downstream decision about features, model choice, or evaluation protocol.
1. **User variability** — how stable is the customer base over time?
2. **Products per user** — what does a typical portfolio look like?
3. **Monthly additions** — how rare is the target?
4. **Churn and net flow** — which products gain and lose holders?
5. **Rating continuum** — what does "months held" actually measure?migration between products?

In [ ]:
# User variability: how many users appear in consecutive months?

import time
t1 = time.time()
users_by_month = (
    curated_lf
    .group_by("fecha_dato")
    .agg(pl.col("ncodpers").unique().alias("users"))
    .sort("fecha_dato")
)

# Self-join con el mes anterior
variability = (
    users_by_month
    .with_columns(
        pl.col("users").shift(1).alias("prev_users")
    )
    .filter(pl.col("prev_users").is_not_null())
    .with_columns([
        pl.col("users").list.len().alias("curr_users"),
        pl.col("prev_users").list.len().alias("prev_users_n"),
        pl.col("users").list.set_intersection("prev_users").list.len().alias("retained"),
    ])
    .with_columns([
        (pl.col("curr_users") - pl.col("retained")).alias("new_users"),
        (pl.col("prev_users_n") - pl.col("retained")).alias("exited"),
        (pl.col("retained") / pl.col("prev_users_n") * 100).round(2).alias("retention_pct"),
        ((pl.col("curr_users") - pl.col("retained")) / pl.col("curr_users") * 100).round(2).alias("new_pct"),
    ])
    .select([
        pl.col("fecha_dato").cast(pl.Utf8).alias("month"),
        "prev_users_n", "curr_users", "retained", "new_users", "exited",
        "retention_pct", "new_pct"
    ])
    .collect()
)

print(variability)

t2 = time.time()
delta = t2 - t1
print(delta)

shape: (16, 8)
┌────────────┬──────────────┬────────────┬──────────┬───────────┬────────┬───────────────┬─────────┐
│ month      ┆ prev_users_n ┆ curr_users ┆ retained ┆ new_users ┆ exited ┆ retention_pct ┆ new_pct │
│ ---        ┆ ---          ┆ ---        ┆ ---      ┆ ---       ┆ ---    ┆ ---           ┆ ---     │
│ str        ┆ u32          ┆ u32        ┆ u32      ┆ u32       ┆ u32    ┆ f64           ┆ f64     │
╞════════════╪══════════════╪════════════╪══════════╪═══════════╪════════╪═══════════════╪═════════╡
│ 2015-02-28 ┆ 625457       ┆ 627394     ┆ 622926   ┆ 4468      ┆ 2531   ┆ 99.6          ┆ 0.71    │
│ 2015-03-28 ┆ 627394       ┆ 629209     ┆ 625094   ┆ 4115      ┆ 2300   ┆ 99.63         ┆ 0.65    │
│ 2015-04-28 ┆ 629209       ┆ 630367     ┆ 626698   ┆ 3669      ┆ 2511   ┆ 99.6          ┆ 0.58    │
│ 2015-05-28 ┆ 630367       ┆ 631957     ┆ 628135   ┆ 3822      ┆ 2232   ┆ 99.65         ┆ 0.6     │
│ 2015-06-28 ┆ 631957       ┆ 632110     ┆ 628603   ┆ 3507      ┆ 3354   ┆ 9

### Reading the variability

Two things stand out:

- **The retention rate is high** (typically 95-98%), meaning most users present one month are also present the next. This is why a model trained on one month generalizes to the next.
- **The new-user rate spikes in July 2015** to around 25%, consistent with a structural change in the customer base. From August 2015 onward, the new-user rate stabilizes at a lower level, but the absolute number of new users each month remains substantial.

The implication: the base is stable enough for temporal validation (April → May should generalize to May → June), but each month introduces enough new users that cold-start handling is non-trivial. A model without demographics cannot score these users.

In [ ]:
# Products per user, by month
PRODUCT_COLS = [c for c in curated_lf.collect_schema().names()
                if c.startswith("ind_") and c.endswith("_ult1") and c != "ind_nuevo"]
products_per_user = (
    curated_lf
    .with_columns(pl.sum_horizontal(PRODUCT_COLS).alias("n_products"))
    .group_by("fecha_dato")
    .agg([
        pl.col("n_products").mean().alias("mean"),
        pl.col("n_products").median().alias("median"),
        (pl.col("n_products") == 0).sum().alias("users_zero"),
        pl.len().alias("n_users"),
        ((pl.col("n_products") == 0).sum() / pl.len() * 100).round(2).alias("pct_zero"),
    ])
    .sort("fecha_dato")
    .collect()
)

print(products_per_user)

shape: (17, 6)
┌────────────┬──────────┬────────┬────────────┬─────────┬──────────┐
│ fecha_dato ┆ mean     ┆ median ┆ users_zero ┆ n_users ┆ pct_zero │
│ ---        ┆ ---      ┆ ---    ┆ ---        ┆ ---     ┆ ---      │
│ date       ┆ f64      ┆ f64    ┆ u32        ┆ u32     ┆ f64      │
╞════════════╪══════════╪════════╪════════════╪═════════╪══════════╡
│ 2015-01-28 ┆ 1.77369  ┆ 1.0    ┆ 17326      ┆ 625457  ┆ 2.77     │
│ 2015-02-28 ┆ 1.773579 ┆ 1.0    ┆ 17127      ┆ 627394  ┆ 2.73     │
│ 2015-03-28 ┆ 1.778152 ┆ 1.0    ┆ 17095      ┆ 629209  ┆ 2.72     │
│ 2015-04-28 ┆ 1.784349 ┆ 1.0    ┆ 17072      ┆ 630367  ┆ 2.71     │
│ 2015-05-28 ┆ 1.772048 ┆ 1.0    ┆ 16919      ┆ 631957  ┆ 2.68     │
│ 2015-06-28 ┆ 1.79162  ┆ 1.0    ┆ 16087      ┆ 632110  ┆ 2.54     │
│ 2015-07-28 ┆ 1.383389 ┆ 1.0    ┆ 204302     ┆ 829817  ┆ 24.62    │
│ 2015-08-28 ┆ 1.365983 ┆ 1.0    ┆ 205042     ┆ 843201  ┆ 24.32    │
│ 2015-09-28 ┆ 1.356195 ┆ 1.0    ┆ 211112     ┆ 865440  ┆ 24.39    │
│ 2015-10-28 ┆ 1.34

### Reading the products-per-user result

The mean products per user drops sharply between June 2015 (1.79) and July 2015 (1.38). Simultaneously, users with zero products jump from 2.5% to 24.6%.

This is a structural break. Roughly 200,000 users entered the bank with almost no products. From August 2015 onward, the distribution stabilizes — mean around 1.33, zero-products fraction around 24-25%.

Two consequences for modeling:

1. **A quarter of the 2016 customer base has no product history to learn from.** Pure collaborative filtering cannot score them — there are no interactions to factorize. This is the primary reason the hybrid model is necessary.
2. **The structural break creates two cohorts.** Users who joined before July 2015 have a different product distribution than those who joined after. A well-designed model must handle both, either through cohort features, seniority features, or training on the mixed base.

In [ ]:
#  Monthly additions: the target we are predicting
add_exprs = [
    ((pl.col(c) == 1) & (pl.col(c).shift(1).over("ncodpers", order_by="fecha_dato") == 0))
    .fill_null(False).cast(pl.UInt8).alias(f"{c}_add")
    for c in PRODUCT_COLS
]


transitions_lf = (
    curated_lf
    .with_columns(add_exprs)
    .with_columns(pl.sum_horizontal([f"{c}_add" for c in PRODUCT_COLS]).alias("n_added"))
    .filter(pl.col("fecha_dato") > pl.col("fecha_dato").min().over("ncodpers"))
)

additions_monthly = (
    transitions_lf
    .group_by("fecha_dato")
    .agg([
        pl.col("n_added").mean().alias("mean_products_added"),
        pl.len().alias("users"),
        (pl.col("n_added") > 0).sum().alias("users_with_addition"),
        ((pl.col("n_added") > 0).sum() / pl.len() * 100).round(2).alias("pct_with_addition"),
        pl.col("n_added").sum().alias("total_additions"),
    ])
    .sort("fecha_dato")
    .collect()
)

print(additions_monthly)


shape: (16, 6)
┌────────────┬───────────────────┬────────┬───────────────────┬──────────────────┬─────────────────┐
│ fecha_dato ┆ mean_products_add ┆ users  ┆ users_with_additi ┆ pct_with_additio ┆ total_additions │
│ ---        ┆ ed                ┆ ---    ┆ on                ┆ n                ┆ ---             │
│ date       ┆ ---               ┆ u32    ┆ ---               ┆ ---              ┆ i64             │
│            ┆ f64               ┆        ┆ u32               ┆ f64              ┆                 │
╞════════════╪═══════════════════╪════════╪═══════════════════╪══════════════════╪═════════════════╡
│ 2015-02-28 ┆ 0.050362          ┆ 622926 ┆ 23782             ┆ 3.82             ┆ 31372           │
│ 2015-03-28 ┆ 0.05128           ┆ 625233 ┆ 25305             ┆ 4.05             ┆ 32062           │
│ 2015-04-28 ┆ 0.048835          ┆ 626944 ┆ 25664             ┆ 4.09             ┆ 30617           │
│ 2015-05-28 ┆ 0.042251          ┆ 628456 ┆ 21422             ┆ 3.41        

In [ ]:
additions_monthly.select(pl.col('pct_with_addition').mean())

pct_with_addition
f64
3.5975


### Reading the additions result
Roughly 3.8% of users add at least one product per month. Range: 2.9% to 5.3%. Seasonality: June 2015 peaks at 5.33% (pre-shift), August 2015 bottoms at 2.94% (dilution), February 2016 peaks at 3.95% (fiscal rhythm).

Three consequences:

1. **Accuracy is not useful.** Predicting "no one adds" gives 96% accuracy and zero value.
2. **The split must be temporal.** Random splitting leaks future information into the past on a seasonal series.
3. **Ranking metrics are required.** MAP@7 measures how well the model puts correct products at the top. Precision@5 with proper masking is the diagnostic equivalent.

## The 96% problem

The competition requires 7 predictions for every one of 929,615 users. Only ~30,000 add anything. The remaining 896,000 contribute AP=0 to MAP@7 regardless of what is predicted. The score is determined entirely by how well the model ranks products for the 3.2% who buy. Feature engineering should focus on "which product will this specific buyer acquire", not "who is going to buy".

In [ ]:
# Additions and churn per product — reveals migration between products
additions_rank = (
    transitions_lf
    .select([pl.col(f"{c}_add").sum().alias(c) for c in PRODUCT_COLS])
    .collect()
    .transpose(include_header=True, column_names=["additions"])
    .sort("additions", descending=True)
)

churn_exprs = [
    ((pl.col(c) == 0) & (pl.col(c).shift(1).over("ncodpers", order_by="fecha_dato") == 1))
    .fill_null(False).cast(pl.UInt8).alias(f"{c}_drop")
    for c in PRODUCT_COLS
]

churn_rank = (
    curated_lf
    .with_columns(churn_exprs)
    .filter(pl.col("fecha_dato") > pl.col("fecha_dato").min().over("ncodpers"))
    .select([pl.col(f"{c}_drop").sum().alias(c) for c in PRODUCT_COLS])
    .collect()
    .transpose(include_header=True, column_names=["drops"])
    .sort("drops", descending=True)
)

net_flow = (
    additions_rank.select(["column", "additions"]).rename({"column": "product"})
    .join(
        churn_rank.select(["column", "drops"]).rename({"column": "product"}),
        on="product",
    )
    .with_columns([
        (pl.col("additions") - pl.col("drops")).alias("net_flow"),
        (pl.col("additions") / pl.col("additions").sum() * 100).round(2).alias("pct_of_additions"),
        (pl.col("additions").cum_sum() / pl.col("additions").sum() * 100).round(2).alias("cumulative_pct"),
    ])
    .sort(["net_flow","pct_of_additions"], descending=[True,True])
)

print(net_flow)

shape: (24, 6)
┌───────────────────┬───────────┬────────┬──────────┬──────────────────┬────────────────┐
│ product           ┆ additions ┆ drops  ┆ net_flow ┆ pct_of_additions ┆ cumulative_pct │
│ ---               ┆ ---       ┆ ---    ┆ ---      ┆ ---              ┆ ---            │
│ str               ┆ i64       ┆ i64    ┆ i64      ┆ f64              ┆ f64            │
╞═══════════════════╪═══════════╪════════╪══════════╪══════════════════╪════════════════╡
│ ind_recibo_ult1   ┆ 153205    ┆ 136594 ┆ 16611    ┆ 27.2             ┆ 27.2           │
│ ind_ecue_fin_ult1 ┆ 26378     ┆ 11035  ┆ 15343    ┆ 4.68             ┆ 93.62          │
│ ind_cno_fin_ult1  ┆ 37187     ┆ 23846  ┆ 13341    ┆ 6.6              ┆ 86.68          │
│ ind_reca_fin_ult1 ┆ 9238      ┆ 11     ┆ 9227     ┆ 1.64             ┆ 100.0          │
│ ind_nom_pens_ult1 ┆ 84768     ┆ 76112  ┆ 8656     ┆ 15.05            ┆ 54.67          │
│ ind_nomina_ult1   ┆ 73800     ┆ 65329  ┆ 8471     ┆ 13.1             ┆ 80.08       

### The concentration

Top-5 products by additions:

| Rank | Product | Additions | % of total | Cumulative % |
|---|---|---|---|---|
| 1 | `ind_recibo_ult1` | 153,205 | 27.2% | 27.2% |
| 2 | `ind_nom_pens_ult1` | 84,768 | 15.1% | 42.3% |
| 3 | `ind_nomina_ult1` | 73,800 | 13.1% | 55.4% |
| 4 | `ind_cco_fin_ult1` | 69,997 | 12.4% | 67.8% |
| 5 | `ind_tjcr_fin_ult1` | 69,311 | 12.3% | **80.1%** |

The top 5 account for **80.1%** of all 563,310 additions. Extending to top-7
(adding `ind_cno_fin_ult1` at 37,187 and `ind_ecue_fin_ult1` at 26,378)
reaches **91.4%**. Top-8 (adding `ind_dela_fin_ult1` at 12,707) reaches
**93.6%**.

The remaining 16 products share 6.4% of the target. `ind_ahor_fin_ult1` has
2 additions in 17 months, `ind_aval_fin_ult1` has 4. These are statistically
too sparse to model — no feature engineering recovers a signal from 2 examples.

The concentration explains why a popularity ranker is competitive. A
recommender that always suggests the top-7 products minus what the user
holds captures **91.4% of the target** by construction. The base rate is
3.8%, the top-7 concentration is 91.4%, and MAP@7 evaluates 7 predictions.
A trivial ranker achieves ~0.018 because those three facts align.

### Migration between products

The net flow reveals a migration pattern that the raw additions number hides:

| Product | Additions | Drops | Net flow | Interpretation |
|---|---|---|---|---|
| `ind_cco_fin_ult1` (traditional account) | 69,997 | 89,644 | **−19,647** | Losing holders |
| `ind_ecue_fin_ult1` (digital account) | 26,378 | 11,035 | **+15,343** | Gaining from migration |
| `ind_cno_fin_ult1` (payroll account) | 37,187 | 23,846 | **+13,341** | Gaining from migration |
| `ind_nomina_ult1` (payroll) | 73,800 | 65,329 | +8,471 | Stable |

The traditional current account loses 19.6k net holders over 17 months. The
digital account and payroll account gain 15.3k and 13.3k respectively. The
numbers are consistent with a single pattern: users are migrating from
`ind_cco_fin_ult1` to `ind_ecue_fin_ult1` or `ind_cno_fin_ult1`.

A user holding the current account but not the digital account is **not** a
cross-sell opportunity. They are a likely migration case. The state signal
"holds A but not B" describes what is changing, not what is available for
acquisition. Models that treat this as opportunity learn spurious correlations:
the user does not need another current account, they need a reason to keep the
one they have.

This is why state-only features mislead. "Holds `ind_cco`" alone is not a
predictive signal. "Holds `ind_cco` but not `ind_ecue`" is — but in the
opposite direction from what naive feature engineering would suggest.

### Two anomalies

**`ind_reca_fin_ult1`**: 9,238 additions and only 11 drops. A product with
9,000 acquisitions and 11 losses is statistically impossible in a real
market. This is a reporting artifact: the product started being tracked
mid-dataset, so "additions" are first appearances in the log, not real
acquisitions. Anyone using `ind_reca` as a target for prediction is modeling
the reporting change, not user behavior.

**`ind_aval_fin_ult1`**: 4 additions in 17 months. Practically dead. With
4 examples, no model can learn anything. The product should be excluded from
any per-product approach.

Both anomalies are visible only because additions and drops are compared
side by side. Additions alone would not have exposed the artifact.



## Rating continuum: what does "months held" measure?

The interaction matrix in most LightFM tutorials uses a "rating" per (user, item) pair. In this dataset, the natural candidate is `rating_frecuencia`: the number of months each user has held each product up to the cutoff.

The intuition is appealing: a product held for 17 months must represent a stronger preference than one held for 2 months. Using `rating_frecuencia` as `sample_weight` should give more weight to stable preferences.

This section tests that intuition. Two analyses: the distribution of the rating across pairs, and the per-product correlation between average holding time and addition frequency. The results are unambiguous: the rating is anti-correlated with the target. Using it as weight biases the model away from what it needs to predict.

A second, related issue emerges from the same data: the interaction matrix built from `rating > 0` includes products the user has abandoned. This creates an inconsistency between training and inference that requires a separate fix.

In [ ]:
from datetime import date

FINAL_CUTOFF = date(2016, 5, 28)
MIN_CUTOFF_FULL = date(2015, 1, 28)
MIN_CUTOFF_RECENT = date(2015, 8, 28)

def build_rating_matrix(min_cutoff, max_cutoff):
    """Rating continuum: months held per user per product within [min_cutoff, max_cutoff]."""
    return (
        curated_lf
        .filter(
            (pl.col("fecha_dato") >= min_cutoff) &
            (pl.col("fecha_dato") <= max_cutoff)
        )
        .group_by("ncodpers")
        .agg([pl.col(c).sum().alias(c) for c in PRODUCT_COLS])
        .unpivot(index="ncodpers", on=PRODUCT_COLS,
                 variable_name="producto", value_name="rating_frecuencia")
        .filter(pl.col("rating_frecuencia") > 0)
        .collect()
    )

rating_full = build_rating_matrix(MIN_CUTOFF_FULL, FINAL_CUTOFF)
rating_recent = build_rating_matrix(MIN_CUTOFF_RECENT, FINAL_CUTOFF)

print("=== Full window (Jan 2015 - May 2016, 17 months) ===")
print(f"Pairs: {len(rating_full):,}")
print(f"Range: {rating_full['rating_frecuencia'].min()} to {rating_full['rating_frecuencia'].max()}")
print(f"Mean: {rating_full['rating_frecuencia'].mean():.2f}")
print(f"Median: {rating_full['rating_frecuencia'].median():.2f}")

print("\n=== Recent window (Aug 2015 - May 2016, 10 months) ===")
print(f"Pairs: {len(rating_recent):,}")
print(f"Range: {rating_recent['rating_frecuencia'].min()} to {rating_recent['rating_frecuencia'].max()}")
print(f"Mean: {rating_recent['rating_frecuencia'].mean():.2f}")
print(f"Median: {rating_recent['rating_frecuencia'].median():.2f}")

=== Full window (Jan 2015 - May 2016, 17 months) ===
Pairs: 1,487,379
Range: 1 to 17
Mean: 13.41
Median: 17.00

=== Recent window (Aug 2015 - May 2016, 10 months) ===
Pairs: 1,404,919
Range: 1 to 10
Mean: 8.60
Median: 10.00


In [ ]:
def rating_distribution(rating_df, name):
    return (
        rating_df
        .group_by("rating_frecuencia")
        .agg([
            pl.len().alias("n_pairs"),
            (pl.len() / len(rating_df) * 100).round(2).alias("pct"),
        ])
        .sort("rating_frecuencia")
        .with_columns(pl.lit(name).alias("window"))
    )

dist_full = rating_distribution(rating_full, "17 months")
dist_recent = rating_distribution(rating_recent, "10 months")

print("=== Distribution (17 months) ===")
print(dist_full)

print("\n=== Distribution (10 months) ===")
print(dist_recent)

=== Distribution (17 months) ===
shape: (17, 4)
┌───────────────────┬─────────┬──────┬───────────┐
│ rating_frecuencia ┆ n_pairs ┆ pct  ┆ window    │
│ ---               ┆ ---     ┆ ---  ┆ ---       │
│ i64               ┆ u32     ┆ f64  ┆ str       │
╞═══════════════════╪═════════╪══════╪═══════════╡
│ 1                 ┆ 57886   ┆ 3.89 ┆ 17 months │
│ 2                 ┆ 43747   ┆ 2.94 ┆ 17 months │
│ 3                 ┆ 46579   ┆ 3.13 ┆ 17 months │
│ 4                 ┆ 32615   ┆ 2.19 ┆ 17 months │
│ 5                 ┆ 33942   ┆ 2.28 ┆ 17 months │
│ 6                 ┆ 36365   ┆ 2.44 ┆ 17 months │
│ 7                 ┆ 34073   ┆ 2.29 ┆ 17 months │
│ 8                 ┆ 40583   ┆ 2.73 ┆ 17 months │
│ 9                 ┆ 36751   ┆ 2.47 ┆ 17 months │
│ 10                ┆ 33745   ┆ 2.27 ┆ 17 months │
│ 11                ┆ 34890   ┆ 2.35 ┆ 17 months │
│ 12                ┆ 28286   ┆ 1.9  ┆ 17 months │
│ 13                ┆ 24828   ┆ 1.67 ┆ 17 months │
│ 14                ┆ 26973   ┆ 1.

### Distribution is concentrated at the maximum

In the full window, **60.8% of pairs have rating = 17**, the maximum. In the recent window, **71.3% have rating = 10**, the maximum. In both cases, most products were held since the first month of the observation window.

This is not a continuous distribution of "preference strength". It is a concentration at "held since the start of the dataset". The rating is effectively pseudo-binary: **the maximum value vs everything else**. Using it as a weight means giving 60-70% of the sample the same weight (the maximum), and distributing the remaining weight across a long tail of short-held products.

Even before considering the correlation with the target, the distribution alone argues against using it as a continuous weight. It does not contain the fine-grained preference information a weight would need to encode.

In [ ]:
def per_product_rating_analysis(rating_df, additions_ref):
    per_product = (
        rating_df
        .group_by("producto")
        .agg([
            pl.len().alias("n_users"),
            pl.col("rating_frecuencia").mean().round(2).alias("avg_months_held"),
            pl.col("rating_frecuencia").median().alias("median_months"),
        ])
    )
    return (
        per_product
        .join(
            additions_ref.rename({"column": "producto", "additions": "total_additions"}),
            on="producto",
        )
        .with_columns([
            pl.col("total_additions").rank(descending=True).cast(pl.Int32).alias("addition_rank"),
        ])
        .sort("avg_months_held", descending=True)
    )

# Additions reference (all-time, no cutoff) — used only for ranking context
additions_all_time = (
    transitions_lf
    .select([pl.col(f"{c}_add").sum().alias(c) for c in PRODUCT_COLS])
    .collect()
    .transpose(include_header=True, column_names=["additions"])
    .sort("additions", descending=True)
)

analysis_full = per_product_rating_analysis(rating_full, additions_all_time)
analysis_recent = per_product_rating_analysis(rating_recent, additions_all_time)

print("=== Per-product analysis (17 months) ===")
print(analysis_full.select([
    "producto", "avg_months_held", "n_users", "total_additions", "addition_rank"
]))

=== Per-product analysis (17 months) ===
shape: (24, 5)
┌───────────────────┬─────────────────┬─────────┬─────────────────┬───────────────┐
│ producto          ┆ avg_months_held ┆ n_users ┆ total_additions ┆ addition_rank │
│ ---               ┆ ---             ┆ ---     ┆ ---             ┆ ---           │
│ str               ┆ f64             ┆ u32     ┆ i64             ┆ i32           │
╞═══════════════════╪═════════════════╪═════════╪═════════════════╪═══════════════╡
│ ind_ctop_fin_ult1 ┆ 16.29           ┆ 108104  ┆ 3882            ┆ 12            │
│ ind_viv_fin_ult1  ┆ 16.07           ┆ 3268    ┆ 70              ┆ 22            │
│ ind_ctpp_fin_ult1 ┆ 15.9            ┆ 37160   ┆ 2420            ┆ 15            │
│ ind_hip_fin_ult1  ┆ 15.82           ┆ 5077    ┆ 75              ┆ 21            │
│ ind_ahor_fin_ult1 ┆ 15.69           ┆ 89      ┆ 2               ┆ 24            │
│ ind_plan_fin_ult1 ┆ 15.52           ┆ 8063    ┆ 618             ┆ 16            │
│ ind_reca_fin_ult1 

In [ ]:
print("=== Per-product analysis (10 months) ===")
print(analysis_recent.select([
    "producto", "avg_months_held", "n_users", "total_additions", "addition_rank"
]))

=== Per-product analysis (10 months) ===
shape: (24, 5)
┌───────────────────┬─────────────────┬─────────┬─────────────────┬───────────────┐
│ producto          ┆ avg_months_held ┆ n_users ┆ total_additions ┆ addition_rank │
│ ---               ┆ ---             ┆ ---     ┆ ---             ┆ ---           │
│ str               ┆ f64             ┆ u32     ┆ i64             ┆ i32           │
╞═══════════════════╪═════════════════╪═════════╪═════════════════╪═══════════════╡
│ ind_hip_fin_ult1  ┆ 9.75            ┆ 4732    ┆ 75              ┆ 21            │
│ ind_ctop_fin_ult1 ┆ 9.74            ┆ 105091  ┆ 3882            ┆ 12            │
│ ind_viv_fin_ult1  ┆ 9.66            ┆ 3132    ┆ 70              ┆ 22            │
│ ind_pres_fin_ult1 ┆ 9.65            ┆ 2081    ┆ 147             ┆ 19            │
│ ind_ctpp_fin_ult1 ┆ 9.58            ┆ 35611   ┆ 2420            ┆ 15            │
│ ind_ahor_fin_ult1 ┆ 9.55            ┆ 83      ┆ 2               ┆ 24            │
│ ind_reca_fin_ult1 

### The rating continuum does not measure affinity

`rating_frecuencia` is the number of months each user has held each product
up to the cutoff. It appears to encode preference: a product held for 17
months must represent a stronger preference than one held for 2. The
distribution and per-product analysis show the opposite.

**Distribution is concentrated at the maximum.** In the full window
(Jan 2015 - May 2016, 17 months), 60.8% of pairs have rating 17. In the
recent window (Aug 2015 - May 2016, 10 months), 71.3% have rating 10, the
new maximum. Shortening the window makes the concentration worse, not
better: fewer months of observation means more pairs hit the ceiling.

The rating is effectively pseudo-binary. It discriminates between "held
since the first observation month" and "held for fewer months". It does
not encode fine-grained preference across the full range.

**Per-product correlation is stable across windows.** Both the 17-month
and 10-month views show the same ordering: products held longest are the
ones added least.

Top-5 by avg_months (17-month window): `ind_ctop_fin_ult1` (16.29),
`ind_viv_fin_ult1` (16.07), `ind_ctpp_fin_ult1` (15.90),
`ind_hip_fin_ult1` (15.82), `ind_ahor_fin_ult1` (15.69). These are
onboarding products. Addition ranks: 12, 22, 15, 21, 24.

Bottom-4 by avg_months (17-month window): `ind_tjcr_fin_ult1` (10.32),
`ind_nomina_ult1` (10.60), `ind_nom_pens_ult1` (10.73),
`ind_recibo_ult1` (11.11). These are the top-4 products by addition
count: ranks 5, 3, 2, 1.

The correlation is not subtle. The four products that dominate the target
have the lowest holding time. The products with the highest holding time
are mature onboarding products that users do not acquire after the
initial signup.

**Three tiers, each with a different mechanism:**

*Old onboarding* (14-17 months avg, ranks 12-24): products users open when
joining the bank. High avg because most holders have held since the first
month. Low additions because most users already have them.

*Cross-sell* (10-14 months avg, ranks 1-8): products added recently by
many users. Medium avg because the holding started recently. High
additions because the cross-sell campaigns are active.

*High-turnover* (3-8 months avg, ranks 10-14): products added and dropped
frequently. Low avg reflects short retention, not low acquisition.

`ind_cco_fin_ult1` is the exception: rank #4 in additions but classified
in the "old onboarding" tier (14.37 months avg). It is both mature
(most users have held it for years) and continues to attract new holders
(through migration from other accounts). The net flow analysis showed
it loses 19.6k holders net, migrating to `ind_ecue` and `ind_cno`.

**Why the weight hurts.** Using `rating_frecuencia` as `sample_weight`
gives more influence to pairs with high month counts. Those pairs are
dominated by old onboarding products held by stable users. Those users
are not the target: they have held their products for years and are not
adding anything. WARP learns to rank old products accurately for stable
users, at the cost of ranking cross-sell products for users in
acquisition mode.

Empirical confirmation: lower performance with this weights,the difference is small but consistent with the analysis.

**What the rating does NOT measure.** It is not affinity, preference, or
engagement. It measures the age of the product in the user's portfolio.
That age is a function of (a) when the product was introduced, (b) when
the user joined the bank, and (c) how long the user has kept the product.
Any of the three can dominate. Using the rating as a weight assumes it
measures (c) alone, which the data does not support.

## Interaction matrix: an empirical question, not a theoretical one

Two constructions are possible:

**Option A — rating > 0 over the full window.** Sum all months held. Include a
pair if the user held the product at any point up to the cutoff.

**Option C — binary state from the cutoff month.** Include a pair if the user
holds the product at May 2016 specifically.

Theoretical reasoning suggests Option C is cleaner. Option A includes abandoned
products as positives, which conflicts with the inference-time masking (which
masks only current holdings). A user who closed their current account in July
2015 appears in the Option A matrix as "has current account", even though the
masking treats them as not having it.

This theoretical argument was tested empirically. It does not hold.

| Construction | Leaderboard Score |
|---|---|
| Option A (rating > 0, full history) | **0.02491** |
| Option C (state-based, no abandoned) | 0.02392 |
| Delta | **−0.001** |

Option A wins by a meaningful margin. The theoretical cleanliness of Option C
does not translate to a better score.

### Why Option A wins

The abandoned pairs are not noise. They are historical signal. A user who held
`ind_cco` for six months before closing it contributes to the co-occurrence
pattern: "users with this profile held current account". The user embedding
learns that pattern. When LightFM combines the user embedding with the
demographic features (`user_feat_matrix`), the historical signal enriches the
current-state signal. The two together are more informative than current state
alone.

Option C throws away this historical signal in exchange for theoretical
alignment. The trade-off is net negative: the 5.5% fewer interaction pairs cost
more in embedding quality than the abandoned-product noise saves.

The lesson: **theory without empirical validation is a hypothesis, not a
conclusion.** The argument that Option C is "the correct construction" was
internally consistent but incomplete. It did not anticipate that the model
benefits from historical signal enough to outweigh the "noise" of abandoned
pairs.

### What this means for the pipeline

The final Model 3 uses Option A. The interaction matrix is built from
`rating_frecuencia > 0` over the full history up to cutoff. The inference-time
masking uses current state (May 2016 holdings). The two are different, and that
difference is a feature, not a bug: the model has access to historical signal
during training, and masking ensures only future acquisitions are recommended.

In [ ]:
FINAL_CUTOFF = date(2016, 5, 28)

state_may_full = (
    curated_lf
    .filter(pl.col("fecha_dato") == FINAL_CUTOFF)
    .collect()
)

state_may_products = state_may_full.select(["ncodpers"] + PRODUCT_COLS)

rating_continuo = (
    curated_lf
    .filter(pl.col("fecha_dato") <= FINAL_CUTOFF)
    .group_by("ncodpers")
    .agg([pl.col(c).sum().alias(c) for c in PRODUCT_COLS])
    .unpivot(index="ncodpers", on=PRODUCT_COLS,
             variable_name="producto", value_name="rating_frecuencia")
    .filter(pl.col("rating_frecuencia") > 0)
    .collect()
)

print(f"Total pairs: {len(rating_continuo):,}")
print(f"Range: {rating_continuo['rating_frecuencia'].min()} to {rating_continuo['rating_frecuencia'].max()}")
print(f"Mean: {rating_continuo['rating_frecuencia'].mean():.2f}")
print(f"Median: {rating_continuo['rating_frecuencia'].median():.2f}")

dist = (
    rating_continuo
    .group_by("rating_frecuencia")
    .agg([
        pl.len().alias("n_pairs"),
        (pl.len() / len(rating_continuo) * 100).round(2).alias("pct"),
    ])
    .sort("rating_frecuencia")
)
print(f"\nDistribution:")
print(dist)

Total pairs: 1,487,379
Range: 1 to 17
Mean: 13.41
Median: 17.00

Distribution:
shape: (17, 3)
┌───────────────────┬─────────┬──────┐
│ rating_frecuencia ┆ n_pairs ┆ pct  │
│ ---               ┆ ---     ┆ ---  │
│ i64               ┆ u32     ┆ f64  │
╞═══════════════════╪═════════╪══════╡
│ 1                 ┆ 57886   ┆ 3.89 │
│ 2                 ┆ 43747   ┆ 2.94 │
│ 3                 ┆ 46579   ┆ 3.13 │
│ 4                 ┆ 32615   ┆ 2.19 │
│ 5                 ┆ 33942   ┆ 2.28 │
│ 6                 ┆ 36365   ┆ 2.44 │
│ 7                 ┆ 34073   ┆ 2.29 │
│ 8                 ┆ 40583   ┆ 2.73 │
│ 9                 ┆ 36751   ┆ 2.47 │
│ 10                ┆ 33745   ┆ 2.27 │
│ 11                ┆ 34890   ┆ 2.35 │
│ 12                ┆ 28286   ┆ 1.9  │
│ 13                ┆ 24828   ┆ 1.67 │
│ 14                ┆ 26973   ┆ 1.81 │
│ 15                ┆ 27971   ┆ 1.88 │
│ 16                ┆ 43843   ┆ 2.95 │
│ 17                ┆ 904302  ┆ 60.8 │
└───────────────────┴─────────┴──────┘


In [ ]:
interactions_from_state = (
    state_may_products
    .unpivot(index="ncodpers", on=PRODUCT_COLS,
             variable_name="producto", value_name="held")
    .filter(pl.col("held") == 1)
    .select(["ncodpers", "producto"])
    .with_columns(pl.col("ncodpers").cast(pl.Int64))
)

print(f"Pairs from state (Option C): {len(interactions_from_state):,}")
print(f"Pairs from rating > 0 (Option A): {len(rating_continuo):,}")
print(f"Difference: {len(rating_continuo) - len(interactions_from_state):,} abandoned pairs excluded")

held_may = {
    row["ncodpers"]: {c for c in PRODUCT_COLS if row[c] == 1}
    for row in state_may_products.iter_rows(named=True)
}

print(f"Held lookup: {len(held_may):,} users")
print(f"Cold-start users: {sum(1 for h in held_may.values() if not h):,}")

Pairs from state (Option C): 1,240,538
Pairs from rating > 0 (Option A): 1,487,379
Difference: 246,841 abandoned pairs excluded
Held lookup: 931,453 users
Cold-start users: 234,914


## Trajectory features

The rating `rating_frecuencia` was attempting to encode "how long has the user held this product". The per-product analysis showed this is anti-correlated with the target, but the underlying signal — whether the user is in acquisition mode — is real and valuable.

That signal is best captured at the user level, not the user-product level. A user who added three products in the last six months is in acquisition mode regardless of which specific products.

Features added:

- **`additions_last_3m`**: total products added in the last 3 months.
- **`additions_last_6m`**: same over 6 months.
- **`additions_lifetime`**: total products added over the entire history.
- **`days_since_last_addition`**: recency. Low value means recent activity. 9999 for users who never added anything.

Computed per user up to the cutoff, joined as columns. They become feature strings in the LightFM user features, not matrix entries.

In [ ]:
def build_trajectory_features(lf: pl.LazyFrame, cutoff_date: date) -> pl.DataFrame:
    recent_3m_cutoff = (
        cutoff_date.replace(month=cutoff_date.month - 3)
        if cutoff_date.month > 3
        else date(cutoff_date.year - 1, cutoff_date.month + 9, cutoff_date.day)
    )
    recent_6m_cutoff = (
        cutoff_date.replace(month=cutoff_date.month - 6)
        if cutoff_date.month > 6
        else date(cutoff_date.year - 1, cutoff_date.month + 6, cutoff_date.day)
    )

    transitions_up_to_cutoff = lf.filter(pl.col("fecha_dato") <= cutoff_date)

    return (
        transitions_up_to_cutoff
        .group_by("ncodpers")
        .agg([
            pl.col("n_added").filter(pl.col("fecha_dato") >= recent_3m_cutoff)
                .sum().fill_null(0).alias("additions_last_3m"),
            pl.col("n_added").filter(pl.col("fecha_dato") >= recent_6m_cutoff)
                .sum().fill_null(0).alias("additions_last_6m"),
            pl.col("n_added").sum().fill_null(0).alias("additions_lifetime"),
            (cutoff_date - pl.col("fecha_dato").filter(pl.col("n_added") > 0).max())
                .dt.total_days().alias("days_since_last_addition"),
        ])
        .with_columns([
            pl.col("days_since_last_addition").fill_null(9999).alias("days_since_last_addition"),
        ])
        .collect()
    )


trajectory_features_may = build_trajectory_features(transitions_lf, FINAL_CUTOFF)
print(f"Trajectory: {len(trajectory_features_may):,} users")
print(trajectory_features_may.head(5))

Trajectory: 946,173 users
shape: (5, 5)
┌──────────┬───────────────────┬───────────────────┬────────────────────┬──────────────────────────┐
│ ncodpers ┆ additions_last_3m ┆ additions_last_6m ┆ additions_lifetime ┆ days_since_last_addition │
│ ---      ┆ ---               ┆ ---               ┆ ---                ┆ ---                      │
│ u32      ┆ i64               ┆ i64               ┆ i64                ┆ i64                      │
╞══════════╪═══════════════════╪═══════════════════╪════════════════════╪══════════════════════════╡
│ 1047770  ┆ 0                 ┆ 0                 ┆ 0                  ┆ 9999                     │
│ 1017211  ┆ 0                 ┆ 0                 ┆ 0                  ┆ 9999                     │
│ 552772   ┆ 0                 ┆ 0                 ┆ 0                  ┆ 9999                     │
│ 1464688  ┆ 0                 ┆ 0                 ┆ 1                  ┆ 213                      │
│ 219343   ┆ 0                 ┆ 0                 

## Custom transformers

Five transformers for the pipeline. Each does one job, compatible with scikit-learn's `Pipeline`, accepts Polars DataFrames natively.

- **Flagnan3Polars** — imputation with median/mode, plus `_NA` flag per feature
- **RareLabelPolars** — groups categories with frequency < tol into "Rare"
- **Flagout2Polars** — clips continuous extremes at 3σ, adds `_out` flag
- **OrdinalEncPolars** — sorted integers for categoricals
- **DropConstantFeaturesPolars** — removes single-value columns

## Model 1: Naive popularity

Rank products by historical additions. Mask products the user already holds. Return top-7. Establishes the floor.

In [ ]:
def build_popularity_until(curated_lf, cutoff_date):
    filtered = curated_lf.filter(pl.col("fecha_dato") <= cutoff_date)
    add_exprs_local = [
        ((pl.col(c) == 1) & (pl.col(c).shift(1).over("ncodpers", order_by="fecha_dato") == 0))
        .fill_null(False).cast(pl.UInt8).alias(f"{c}_add")
        for c in PRODUCT_COLS
    ]
    return (
        filtered
        .with_columns(add_exprs_local)
        .filter(pl.col("fecha_dato") > pl.col("fecha_dato").min().over("ncodpers"))
        .select([pl.col(f"{c}_add").sum().alias(c) for c in PRODUCT_COLS])
        .collect()
        .transpose(include_header=True, column_names=["additions"])
        .sort("additions", descending=True)
    )


pop_rank_may = build_popularity_until(curated_lf, FINAL_CUTOFF)
top7_pop = pop_rank_may.head(7)["column"].to_list()
pop_string = " ".join(top7_pop)

print(f"Popularity top-7: {top7_pop}")

sample = pd.read_csv(SAMPLE_SUB)
test_user_ids = sample["ncodpers"].tolist()

pop_rows = []
for uid in test_user_ids:
    held = held_may.get(uid, set())
    recs = [p for p in top7_pop if p not in held][:7]
    while len(recs) < 7:
        for p in pop_rank_may["column"].to_list():
            if p not in recs:
                recs.append(p)
                if len(recs) == 7:
                    break
    pop_rows.append({"ncodpers": uid, "added_products": " ".join(recs[:7])})

pop_sub = pd.DataFrame(pop_rows).sort_values("ncodpers").reset_index(drop=True)
pop_sub.to_csv("/content/submission_popularity.csv", index=False)
print(f"Popularity submission: {len(pop_sub):,} rows")

Popularity top-7: ['ind_recibo_ult1', 'ind_nom_pens_ult1', 'ind_nomina_ult1', 'ind_cco_fin_ult1', 'ind_tjcr_fin_ult1', 'ind_cno_fin_ult1', 'ind_ecue_fin_ult1']
Popularity submission: 929,615 rows


## Model 2: Collaborative only

LightFM factorizes users and products from the interaction matrix alone. No demographics, no other features.

Isolates the collaborative signal. The expected score (0.0215) is 17% above popularity but well below the hybrid. The gap is the 25% cold-start cohort: without demographics, LightFM cannot score users with no product history.

WARP loss, 40 components, 20 epochs. No sample weights, from rating > 0 over full history.



In [ ]:
interactions_collab = (
    rating_continuo
    .select(["ncodpers", "producto"])
    .with_columns(pl.col("ncodpers").cast(pl.Utf8).alias("user_str"))
    .filter(pl.col("producto").is_in(PRODUCT_COLS))
)
# option C
# interactions_collab = (
#     interactions_from_state
#     .select(["ncodpers", "producto"])
#     .with_columns(pl.col("ncodpers").cast(pl.Utf8).alias("user_str"))
# )

users_collab = interactions_collab["user_str"].unique().to_list()

lfm_collab = LightFMDataset()
lfm_collab.fit(
    users=users_collab,
    items=PRODUCT_COLS,
)

interactions_matrix_collab, _ = lfm_collab.build_interactions(
    (u, p) for u, p in interactions_collab.select(["user_str", "producto"]).iter_rows()
)

model_collab = LightFM(
    loss="warp",
    no_components=40,
    learning_rate=0.05,
    random_state=42,
)

model_collab.fit(
    interactions_matrix_collab,
    epochs=20,
    num_threads=4,
    verbose=True,
)

Epoch: 100%|██████████| 20/20 [00:28<00:00,  1.43s/it]


In [ ]:

user_map_c, _, item_map_c, _ = lfm_collab.mapping()
inv_item_map_c = {v: k for k, v in item_map_c.items()}
n_items = len(item_map_c)
all_items = np.arange(n_items)

known_uids = [uid for uid in test_user_ids if str(uid) in user_map_c]
known_internal = np.array([user_map_c[str(uid)] for uid in known_uids], dtype=np.int32)

BATCH = 4096
scores_all = np.empty((len(known_uids), n_items), dtype=np.float32)

for start in range(0, len(known_uids), BATCH):
    end = min(start + BATCH, len(known_uids))
    batch_users = known_internal[start:end]

    user_ids_batch = np.repeat(batch_users, n_items)
    item_ids_batch = np.tile(all_items, len(batch_users))

    scores_all[start:end] = model_collab.predict(
        user_ids_batch,
        item_ids_batch,
        num_threads=4
    ).reshape(len(batch_users), n_items)

uid_to_idx = {uid: i for i, uid in enumerate(known_uids)}
pop_list = pop_rank_may["column"].to_list()
pop_fallback = pop_list[:7]

collab_rows = []
n_scored_c = 0
n_fallback_c = 0

for uid in test_user_ids:
    if uid in uid_to_idx:
        scores = scores_all[uid_to_idx[uid]].copy()

        # Mask products already held
        for item_name in held_may.get(uid, set()):
            if item_name in item_map_c:
                scores[item_map_c[item_name]] = -np.inf

        top_idx = np.argpartition(-scores, 6)[:7]
        top_idx = top_idx[np.argsort(-scores[top_idx])]
        recs = [inv_item_map_c[j] for j in top_idx if np.isfinite(scores[j])]
        n_scored_c += 1
    else:
        recs = list(pop_fallback)
        n_fallback_c += 1

    # Fill to 7 if needed
    for p in pop_list:
        if len(recs) >= 7:
            break
        if p not in recs:
            recs.append(p)

    collab_rows.append({
        "ncodpers": uid,
        "added_products": " ".join(recs[:7])
    })

collab_sub = pd.DataFrame(collab_rows).sort_values("ncodpers").reset_index(drop=True)
collab_sub.to_csv("/content/submission_collaborative.csv", index=False)

print(f"Scored by model: {n_scored_c:,}")
print(f"Popularity fallback (cold-start): {n_fallback_c:,}")
print(f"Total rows: {len(collab_sub):,}")

Scored by model: 715,262
Popularity fallback (cold-start): 214,353
Total rows: 929,615


## Model 3 vs Model 4: with or without trajectory features

Two configurations share the same pipeline structure:

**Model 3 — no trajectory.** Demographics only. Continuous features: age, renta, antiguedad_dias. This is the configuration that scored **0.02493** on the leaderboard.

**Model 4 — with trajectory.** Demographics plus four rolling addition features: additions_last_3m, additions_last_6m, additions_lifetime, days_since_last_addition. Continuous features: the three from Model 3 plus the four new ones. This is the pending experiment.

Both use:
- Interaction matrix from `rating > 0` over full history (Option A)
- No sample weights, no binning, ordinal encoding on all categoricals
- WARP, 40 components, 20 epochs

The Option A vs Option C comparison is documented in the "Interaction matrix:
an empirical question" section. Option A wins by 0.001.

The only difference is the presence of the four trajectory columns in the input dataframe and in the `vars_numericas` list passed to `Flagout2Polars`.

Both get submitted to Kaggle. Whatever the leaderboard says is the answer. Only the winner goes through temporal validation.

### Model 3: hybrid pipeline without trajectory

Configuration that scored 0.02493. The dataframe is the May 2016 state with demographic columns only. The pipeline treats `age`, `renta`, `antiguedad_dias` as the continuous features to clip at 3σ.

Two helper functions are defined here and reused by Model 4 and the validation block:

- `fit_pipeline_and_model(df, vars_num, interactions_df)` — fits the pipeline and trains the LightFM model. Returns a bundle with the pipeline, the feature dataframe, the model, the dataset mapping, and the user feature matrix.
- `predict_for_users(bundle, user_ids, held_lookup, pop_list)` — predicts top-7 for the given users, applying masking and popularity fallback.



In [ ]:
FINAL_CUTOFF = date(2016, 5, 28)

state_m3 = (
    curated_lf
    .filter(pl.col("fecha_dato") == FINAL_CUTOFF)
    .collect()
)

demo_candidates = [
    "ncodpers", "fecha_dato", "fecha_alta", "age", "renta",
    "ind_empleado", "pais_residencia", "sexo", "ind_nuevo",
    "indrel", "indrel_1mes", "tiprel_1mes", "indresi", "indext",
    "canal_entrada", "indfall", "cod_prov", "nomprov",
    "ind_actividad_cliente", "segmento",
]
demo_cols = [c for c in demo_candidates if c in state_m3.columns]

df_unificado_m3 = (
    state_m3.select(demo_cols)
    .with_columns([
        (pl.col("fecha_dato") - pl.col("fecha_alta")).dt.total_days().clip(lower_bound=0).alias("antiguedad_dias"),
        pl.col("fecha_dato").dt.month().alias("mes_dato"),
        pl.col("fecha_alta").dt.month().alias("alta_mes"),
        pl.col("fecha_alta").dt.year().alias("alta_anio"),
    ])
    .with_columns([
        (pl.col("mes_dato") * (2 * np.pi / 12)).sin().alias("mes_dato_sin").cast(pl.Float64, strict=False),
        (pl.col("mes_dato") * (2 * np.pi / 12)).cos().alias("mes_dato_cos").cast(pl.Float64, strict=False),
        (pl.col("alta_mes") * (2 * np.pi / 12)).sin().alias("alta_mes_sin").cast(pl.Float64, strict=False),
        (pl.col("alta_mes") * (2 * np.pi / 12)).cos().alias("alta_mes_cos").cast(pl.Float64, strict=False),
    ])
    .with_columns([
        pl.col("age").cast(pl.Float64, strict=False),
        pl.col("renta").cast(pl.Float64, strict=False),
    ])
    .drop(["fecha_dato", "fecha_alta", "mes_dato", "alta_mes"])
)

categorical_candidates = [
    "ind_empleado", "pais_residencia", "sexo", "ind_nuevo",
    "indrel", "indrel_1mes", "tiprel_1mes", "indresi", "indext",
    "canal_entrada", "indfall", "cod_prov", "nomprov",
    "ind_actividad_cliente", "segmento",
]
vars_categoricas_m3 = [c for c in categorical_candidates if c in df_unificado_m3.columns]

df_unificado_m3 = df_unificado_m3.with_columns(
    pl.col(vars_categoricas_m3).cast(pl.String, strict=False)
)

# Dynamic classification
cat_low_m3, cat_high_m3, num_m3, disc_m3, cont_m3 = categorizar_todo_polars(
    df_unificado_m3,
    max_cat=20, n_dis=30,
    excluir=["ncodpers", "target_bin", "unix_timestamp", "release_date",
             "video_release_date", "user_id", "movie_id"],
)

var_all_m3 = cat_low_m3 + cat_high_m3 + num_m3
cat_high2_m3 = [c for c in cat_high_m3 if c != "title"]
var_all_notext=[ var for var in var_all_m3  if var not in ["title"]]
print(f"cat_low:     {len(cat_low_m3)}")
print(f"cat_high:    {len(cat_high_m3)}")
print(f"num_all:     {len(num_m3)}")
print(f"discrete:    {len(disc_m3)}")
print(f"continuous:  {len(cont_m3)} -> {cont_m3}")

pipe_m3 = Pipeline([
    ("na_imputer", Flagnan3Polars(var=var_all_m3)),
    ("rare_labels", RareLabelPolars(var=cat_low_m3 + cat_high_m3 + disc_m3, tol=0.01)),
    ("out", Flagout2Polars(var=cont_m3, distance=3.0)),
    ("ordinal_encoder", OrdinalEncPolars(var=cat_high2_m3 + cat_low_m3)),
    ("drop", DropConstantFeaturesPolars()),
])

df_train_m3 = pipe_m3.fit_transform(df_unificado_m3)
cyclic_cols = ["mes_dato_sin", "mes_dato_cos", "alta_mes_sin", "alta_mes_cos","alta_anio"]
df_train_m3 = df_train_m3.with_columns([
    pl.col(c).cast(pl.Float64, strict=False) for c in cyclic_cols
    if c in df_train_m3.columns
])
print(f"\nFeatures after pipeline: {len(df_train_m3.columns) - 1}, users: {len(df_train_m3):,}")

cat_low:     11
cat_high:    4
num_all:     8
discrete:    5
continuous:  3 -> ['age', 'renta', 'antiguedad_dias']

Features after pipeline: 26, users: 931,453


In [ ]:
skim(df_train_m3.to_pandas())

── Data Summary ────────────────────────────────────────────────────────────────────────────────────
type       value
-------  -------
Rows      931453
Columns       27

Column type frequency:
           Count
-------  -------
int64         15
float64        6
int32          5
uint32         1


── Variable type: number ───────────────────────────────────────────────────────────────────────────
    name                     na_count    na_count_fr    n_unique       mean         sd         p0         p25        p50        p75       p100  hist
--  ---------------------  ----------  -------------  ----------  ---------  ---------  ---------  ----------  ---------  ---------  ---------  ----------
 0  ncodpers                        0         0           931453   8.8e+05    4.48e+05   1.59e+04    4.84e+05   9.67e+05   1.27e+06   1.55e+06  ▂ ▂  ▂▇▇▇▇
 1  age                             0         0               91  40.1       17.1        2          25         38         50         91.7      

In [ ]:
rating_continuo_m3 = (
    curated_lf
    .filter(pl.col("fecha_dato") <= FINAL_CUTOFF)
    .group_by("ncodpers")
    .agg([pl.col(c).sum().alias(c) for c in PRODUCT_COLS])
    .unpivot(index="ncodpers", on=PRODUCT_COLS,
             variable_name="producto", value_name="rating_frecuencia")
    .filter(pl.col("rating_frecuencia") > 0)
    .with_columns([
        pl.col("ncodpers").cast(pl.Int64),
        pl.col("rating_frecuencia").cast(pl.Float32),
    ])
    .collect()
)

feature_cols_m3 = [c for c in df_train_m3.columns if c != "ncodpers"]

user_rows_m3 = []
for row in df_train_m3.iter_rows(named=True):
    uid = str(int(row["ncodpers"]))
    feats = []
    for col in feature_cols_m3:
        val = row[col]
        if val is None:
            continue
        if col in cyclic_cols and isinstance(val, float):
            feats.append(f"{col}:{round(val, 2)}")
        else:
            feats.append(f"{col}:{val}")
    if feats:
        user_rows_m3.append((uid, feats))

valid_users_m3 = set(uid for uid, _ in user_rows_m3)
rating_filtered_m3 = rating_continuo_m3.filter(
    pl.col("ncodpers").cast(pl.Utf8).is_in(list(valid_users_m3))
)

all_features_m3 = sorted({f for _, feats in user_rows_m3 for f in feats})

lfm_m3 = LightFMDataset()
lfm_m3.fit(
    users=list(valid_users_m3),
    items=PRODUCT_COLS,
    user_features=all_features_m3,
)

interactions_m3, _ = lfm_m3.build_interactions(
    (str(u), p, float(r))
    for u, p, r in rating_filtered_m3.select(
        ["ncodpers", "producto", "rating_frecuencia"]
    ).iter_rows()
)
# interactions_m3, _ = lfm_m3.build_interactions(
#     (str(u), p)
#     for u, p in interactions_from_state.select(["ncodpers", "producto"]).iter_rows()
# )

ufm_m3 = lfm_m3.build_user_features(user_rows_m3)

print(f"Users: {len(valid_users_m3):,}")
print(f"Interactions: {interactions_m3.shape}, nnz={interactions_m3.nnz:,}")
print(f"User features: {ufm_m3.shape}, nnz={ufm_m3.nnz:,}")

model_m3 = LightFM(
    loss="warp",
    no_components=40,
    learning_rate=0.06,
    random_state=42,
)

model_m3.fit(
    interactions_m3,
    user_features=ufm_m3,
    epochs=20,
    num_threads=4,
    verbose=True,
)

Users: 931,453
Interactions: (931453, 24), nnz=1,463,393
User features: (931453, 1452572), nnz=25,135,473


Epoch: 100%|██████████| 20/20 [03:15<00:00,  9.78s/it]


In [ ]:
import joblib

model_m33=joblib.load("/content/model_m3.joblib")

ValueError: EOF: reading array data, expected 262144 bytes got 56566

In [ ]:
sample = pd.read_csv(SAMPLE_SUB)
test_user_ids = sample["ncodpers"].tolist()

state_m3_products = state_m3.select(["ncodpers"] + PRODUCT_COLS)
held_m3 = {
    row["ncodpers"]: {c for c in PRODUCT_COLS if row[c] == 1}
    for row in state_m3_products.iter_rows(named=True)
}

pop_rank_m3 = build_popularity_until(curated_lf, FINAL_CUTOFF)
top7_pop_m3 = pop_rank_m3.head(7)["column"].to_list()
pop_string_m3 = " ".join(top7_pop_m3)

user_map_m3, _, item_map_m3, _ = lfm_m3.mapping()
inv_item_map_m3 = {v: k for k, v in item_map_m3.items()}
all_items_m3 = np.arange(len(item_map_m3))

known_uids_m3 = [uid for uid in test_user_ids if str(uid) in user_map_m3]
known_internal_m3 = np.array([user_map_m3[str(uid)] for uid in known_uids_m3], dtype=np.int32)

BATCH = 4096
scores_all_m3 = np.empty((len(known_uids_m3), len(item_map_m3)), dtype=np.float32)

for start in range(0, len(known_uids_m3), BATCH):
    end = min(start + BATCH, len(known_uids_m3))
    bi = known_internal_m3[start:end]
    ub = np.repeat(bi, len(item_map_m3))
    ib = np.tile(all_items_m3, len(bi))
    scores_all_m3[start:end] = model_m3.predict(
        ub, ib, user_features=ufm_m3, num_threads=4,
    ).reshape(len(bi), len(item_map_m3))

uid_to_idx_m3 = {uid: i for i, uid in enumerate(known_uids_m3)}

rows_m3 = []
n_scored_m3, n_fallback_m3 = 0, 0

for uid in test_user_ids:
    if uid in uid_to_idx_m3:
        scores = scores_all_m3[uid_to_idx_m3[uid]].copy()
        for item in held_m3.get(uid, set()):
            if item in item_map_m3:
                scores[item_map_m3[item]] = -np.inf
        top_idx = np.argpartition(-scores, 6)[:7]
        top_idx = top_idx[np.argsort(-scores[top_idx])]
        recs = [inv_item_map_m3[j] for j in top_idx if np.isfinite(scores[j])]
        n_scored_m3 += 1
    else:
        recs = top7_pop_m3[:7]
        n_fallback_m3 += 1

    for p in pop_rank_m3["column"].to_list():
        if len(recs) >= 7:
            break
        if p not in recs:
            recs.append(p)

    rows_m3.append({"ncodpers": uid, "added_products": " ".join(recs[:7])})

submission_m3 = pd.DataFrame(rows_m3).sort_values("ncodpers").reset_index(drop=True)
submission_m3.to_csv("/content/submission_model_3_v2.csv", index=False)

assert len(submission_m3) == len(sample)
assert set(submission_m3["ncodpers"]) == set(test_user_ids)
assert submission_m3["ncodpers"].duplicated().sum() == 0
assert all(len(r.split()) == 7 for r in submission_m3["added_products"])

print(f"Scored: {n_scored_m3:,}")
print(f"Fallback: {n_fallback_m3:,}")
print(f"Rows: {len(submission_m3):,}")

Scored: 929,615
Fallback: 0
Rows: 929,615


In [ ]:
# Persists the winning Model 3 v2 pipeline and all artifacts required for inference.
# Target: reproduce Kaggle private score 0.02493 from a fresh session.
# No state outside of the returned bundle is needed.

import json
import pickle
import shutil
from pathlib import Path
import joblib
from scipy.sparse import save_npz


BUNDLE_VERSION = "model3_v2_kaggle_0.02493"

bundle_dir="/content/model_v2"
expected_score=0.025
out = Path(bundle_dir)
if out.exists():
    shutil.rmtree(out)
out.mkdir(parents=True)

# 1. Fitted transformation pipeline
# with open(out / "pipe_m3.pkl", "wb") as f:
#     pickle.dump(pipe_m3, f, protocol=pickle.HIGHEST_PROTOCOL)
# print(f"[OK] pipe_m3: {len(df_train_m3.columns) - 1} output features")

# # 2. Trained LightFM model
# with open(out / "model_m3.pkl", "wb") as f:
#     pickle.dump(model_m3, f, protocol=pickle.HIGHEST_PROTOCOL)
# print(f"[OK] model_m3: loss={model_m3.loss} components={model_m3.no_components}")

# 3. LightFM Dataset (index maps)
with open(out / "lfm_m3.pkl", "wb") as f:
    pickle.dump(lfm_m3, f, protocol=pickle.HIGHEST_PROTOCOL)
user_map, _, item_map, _ = lfm_m3.mapping()
print(f"[OK] lfm_m3: users={len(user_map):,} items={len(item_map)}")

joblib.dump(pipe_m3,  out / "pipe_m3.joblib",  compress=3)
joblib.dump(model_m3, out / "model_m3.joblib", compress=3)

# 4. Sparse matrices
save_npz(out / "ufm_m3.npz", ufm_m3)
save_npz(out / "interactions_m3.npz", interactions_m3)
print(f"[OK] ufm_m3.npz: {ufm_m3.shape} nnz={ufm_m3.nnz:,}")
print(f"[OK] interactions_m3.npz: {interactions_m3.shape} nnz={interactions_m3.nnz:,}")

# 5. Post-pipeline training features and popularity ranking
df_train_m3.write_parquet(out / "df_train_m3.parquet")
pop_rank_m3.write_parquet(out / "pop_rank_m3.parquet")
print(f"[OK] df_train_m3.parquet: {df_train_m3.shape}")
print(f"[OK] pop_rank_m3.parquet: {pop_rank_m3.shape}")

# 6. Held-products state for inference masking
with open(out / "held_m3.pkl", "wb") as f:
    pickle.dump(held_m3, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f"[OK] held_m3: {len(held_m3):,} users")

# 7. Raw CSV curation plan (optional for new data)
with open(out / "plan_polars.pkl", "wb") as f:
    pickle.dump(plan_polars, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f"[OK] plan_polars: {len(plan_polars['transforms'])} transforms")

# 8. Metadata
metadata = {
    "version": BUNDLE_VERSION,
    "expected_private_score": expected_score,
    "final_cutoff": str(FINAL_CUTOFF),
    "product_cols": PRODUCT_COLS,
    "demo_cols": demo_cols,
    "vars_categoricas": vars_categoricas_m3,
    "cyclic_cols": [
        "mes_dato_sin", "mes_dato_cos",
        "alta_mes_sin", "alta_mes_cos", "alta_anio",
    ],
    "feature_cols": [c for c in df_train_m3.columns if c != "ncodpers"],
    "top7_popularity": top7_pop_m3,
    "pop_list": pop_rank_m3["column"].to_list(),
    "n_users_train": len(df_train_m3),
    "n_items": len(PRODUCT_COLS),
    "n_user_features": int(ufm_m3.shape[1]),
    "model": {
        "loss": "warp",
        "no_components": 40,
        "learning_rate": 0.05,
        "epochs": 20,
        "random_state": 42,
    },
    "interaction_construction": "Option A: rating > 0 over full history up to cutoff",
    "inference_note": (
        "Predictions use ufm_m3 (training features from May 2016). "
        "Users absent from user_map fall back to popularity ranking. "
        "The test CSV does not need to be processed to reproduce the submission."
    ),
}
with open(out / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print("[OK] metadata.json")

# 9. Zip archive
zip_path = Path(shutil.make_archive(str(out), "zip", out))
size_mb = zip_path.stat().st_size / 1e6
print(f"[OK] archive: {zip_path} ({size_mb:.1f} MB)")


[OK] lfm_m3: users=931,453 items=24
[OK] ufm_m3.npz: (931453, 1452572) nnz=25,135,473
[OK] interactions_m3.npz: (931453, 24) nnz=1,463,393
[OK] df_train_m3.parquet: (931453, 27)
[OK] pop_rank_m3.parquet: (24, 2)
[OK] held_m3: 931,453 users
[OK] plan_polars: 45 transforms
[OK] metadata.json
[OK] archive: /content/model_v2.zip (369.2 MB)


In [ ]:
import joblib, os
import numpy as np

# 1. Tamaño por artefacto en el bundle
for f in sorted(os.listdir("/content/model_v2")):
    p = os.path.join("/content/model_v2", f)
    print(f"{f:<30} {os.path.getsize(p)/1e6:>8.1f} MB")

# 2. Breakdown interno del modelo LightFM
model = joblib.load("/content/model_v2/model_m3.joblib")
for attr in ["user_embeddings", "user_biases", "item_embeddings",
             "item_biases", "user_feature_embeddings", "item_feature_embeddings"]:
    if hasattr(model, attr):
        arr = getattr(model, attr)
        if arr is not None:
            mb = arr.nbytes / 1e6
            print(f"{attr:<28} {str(arr.shape):<20} {mb:>7.1f} MB  {arr.dtype}")

df_train_m3.parquet                 9.9 MB
held_m3.pkl                        10.4 MB
interactions_m3.npz                 4.6 MB
lfm_m3.pkl                         34.9 MB
metadata.json                       0.0 MB
model_m3.joblib                   324.8 MB
pipe_m3.joblib                      0.0 MB
plan_polars.pkl                     0.0 MB
pop_rank_m3.parquet                 0.0 MB
ufm_m3.npz                         16.5 MB
user_embeddings              (1452572, 40)          232.4 MB  float32
user_biases                  (1452572,)               5.8 MB  float32
item_embeddings              (24, 40)                 0.0 MB  float32
item_biases                  (24,)                    0.0 MB  float32


## Model 4: hybrid pipeline with trajectory features

Same pipeline structure as Model 3. One addition: the four trajectory columns are joined to the state before feature engineering.

The trajectory features are numeric. `categorizar_todo_polars` decides their fate based on cardinality:
- `additions_last_3m`, `additions_last_6m`, `additions_lifetime` (low unique counts) → `discrete_final`, get rare label grouping
- `days_since_last_addition` (high unique counts) → `cont_final`, gets clipped at 3σ

The pipeline treats them exactly like any other numeric feature. The only difference from Model 3 is four extra columns in the input and their classification.

In [ ]:
trajectory_m4 = build_trajectory_features(transitions_lf, FINAL_CUTOFF)

state_m4 = (
    state_m3.join(trajectory_m4, on="ncodpers", how="left")
    .with_columns([
        pl.col("additions_last_3m").fill_null(0),
        pl.col("additions_last_6m").fill_null(0),
        pl.col("additions_lifetime").fill_null(0),
        pl.col("days_since_last_addition").fill_null(9999),
    ])
)

traj_cols = ["additions_last_3m", "additions_last_6m", "additions_lifetime", "days_since_last_addition"]

state_m4 = state_m4.with_columns([
    pl.col(c).cast(pl.Float64) for c in traj_cols
])

df_unificado_m4 = (
    state_m4.select(demo_cols + traj_cols)
    .with_columns([
        (pl.col("fecha_dato") - pl.col("fecha_alta")).dt.total_days().clip(lower_bound=0).alias("antiguedad_dias"),
        pl.col("fecha_dato").dt.month().alias("mes_dato"),
        pl.col("fecha_alta").dt.month().alias("alta_mes"),
        pl.col("fecha_alta").dt.year().alias("alta_anio"),
    ])
    .with_columns([
        (pl.col("mes_dato") * (2 * np.pi / 12)).sin().alias("mes_dato_sin").cast(pl.Float64, strict=False),
        (pl.col("mes_dato") * (2 * np.pi / 12)).cos().alias("mes_dato_cos").cast(pl.Float64, strict=False),
        (pl.col("alta_mes") * (2 * np.pi / 12)).sin().alias("alta_mes_sin").cast(pl.Float64, strict=False),
        (pl.col("alta_mes") * (2 * np.pi / 12)).cos().alias("alta_mes_cos").cast(pl.Float64, strict=False),
    ])
    .with_columns([
        pl.col("age").cast(pl.Float64, strict=False),
        pl.col("renta").cast(pl.Float64, strict=False),
    ])
    .drop(["fecha_dato", "fecha_alta", "mes_dato", "alta_mes"])
)

# Cast only predefined categoricals to string; trajectory stays numeric
df_unificado_m4 = df_unificado_m4.with_columns(
    pl.col(vars_categoricas_m3).cast(pl.String, strict=False)
)

cat_low_m4, cat_high_m4, num_m4, disc_m4, cont_m4 = categorizar_todo_polars(
    df_unificado_m4,
    max_cat=20, n_dis=30,
    excluir=["ncodpers", "target_bin", "unix_timestamp", "release_date",
             "video_release_date", "user_id", "movie_id"],
)

var_all_m4 = cat_low_m4 + cat_high_m4 + num_m4
cat_high2_m4 = [c for c in cat_high_m4 if c != "title"]

print(f"cat_low:     {len(cat_low_m4)}")
print(f"cat_high:    {len(cat_high_m4)}")
print(f"num_all:     {len(num_m4)}")
print(f"discrete:    {len(disc_m4)} -> {disc_m4}")
print(f"continuous:  {len(cont_m4)} -> {cont_m4}")

pipe_m4 = Pipeline([
    ("na_imputer", Flagnan3Polars(var=var_all_m4)),
    ("rare_labels", RareLabelPolars(var=cat_low_m4 + cat_high_m4 + disc_m4, tol=0.01)),
    ("out", Flagout2Polars(var=cont_m4, distance=3.0)),
    ("ordinal_encoder", OrdinalEncPolars(var=cat_high2_m4 + cat_low_m4)),
    ("drop", DropConstantFeaturesPolars()),
])

df_train_m4 = pipe_m4.fit_transform(df_unificado_m4)

df_train_m4 = df_train_m4.with_columns([
    pl.col(c).cast(pl.Float64, strict=False) for c in cyclic_cols
    if c in df_train_m4.columns
])

print(f"\nFeatures after pipeline: {len(df_train_m4.columns) - 1}, users: {len(df_train_m4):,}")

cat_low:     11
cat_high:    4
num_all:     12
discrete:    9 -> ['additions_last_3m', 'additions_last_6m', 'additions_lifetime', 'days_since_last_addition', 'alta_anio', 'mes_dato_sin', 'mes_dato_cos', 'alta_mes_sin', 'alta_mes_cos']
continuous:  3 -> ['age', 'renta', 'antiguedad_dias']

Features after pipeline: 30, users: 931,453


In [ ]:
feature_cols_m4 = [c for c in df_train_m4.columns if c != "ncodpers"]

user_rows_m4 = []
for row in df_train_m4.iter_rows(named=True):
    uid = str(int(row["ncodpers"]))
    feats = []
    for col in feature_cols_m4:
        val = row[col]
        if val is None:
            continue
        if col in cyclic_cols and isinstance(val, float):
            feats.append(f"{col}:{round(val, 2)}")
        else:
            feats.append(f"{col}:{val}")
    if feats:
        user_rows_m4.append((uid, feats))

valid_users_m4 = set(uid for uid, _ in user_rows_m4)
rating_filtered_m4 = rating_continuo_m3.filter(
    pl.col("ncodpers").cast(pl.Utf8).is_in(list(valid_users_m4))
)

all_features_m4 = sorted({f for _, feats in user_rows_m4 for f in feats})

lfm_m4 = LightFMDataset()
lfm_m4.fit(
    users=list(valid_users_m4),
    items=PRODUCT_COLS,
    user_features=all_features_m4,
)

interactions_m4, _ = lfm_m4.build_interactions(
    (str(u), p, float(r))
    for u, p, r in rating_filtered_m4.select(
        ["ncodpers", "producto", "rating_frecuencia"]
    ).iter_rows()
)
ufm_m4 = lfm_m4.build_user_features(user_rows_m4)

print(f"Users: {len(valid_users_m4):,}")
print(f"Interactions: {interactions_m4.shape}, nnz={interactions_m4.nnz:,}")
print(f"User features: {ufm_m4.shape}, nnz={ufm_m4.nnz:,}")

model_m4 = LightFM(
    loss="warp",
    no_components=40,
    learning_rate=0.06,
    random_state=42,
)

model_m4.fit(
    interactions_m4,
    user_features=ufm_m4,
    epochs=20,
    num_threads=4,
    verbose=True,
)

Users: 931,453
Interactions: (931453, 24), nnz=1,463,393
User features: (931453, 1452599), nnz=28,861,285


Epoch: 100%|██████████| 20/20 [03:20<00:00, 10.02s/it]


In [ ]:
held_m4 = {
    row["ncodpers"]: {c for c in PRODUCT_COLS if row[c] == 1}
    for row in state_m3_products.iter_rows(named=True)
}

user_map_m4, _, item_map_m4, _ = lfm_m4.mapping()
inv_item_map_m4 = {v: k for k, v in item_map_m4.items()}
all_items_m4 = np.arange(len(item_map_m4))

known_uids_m4 = [uid for uid in test_user_ids if str(uid) in user_map_m4]
known_internal_m4 = np.array([user_map_m4[str(uid)] for uid in known_uids_m4], dtype=np.int32)

scores_all_m4 = np.empty((len(known_uids_m4), len(item_map_m4)), dtype=np.float32)

for start in range(0, len(known_uids_m4), BATCH):
    end = min(start + BATCH, len(known_uids_m4))
    bi = known_internal_m4[start:end]
    ub = np.repeat(bi, len(item_map_m4))
    ib = np.tile(all_items_m4, len(bi))
    scores_all_m4[start:end] = model_m4.predict(
        ub, ib, user_features=ufm_m4, num_threads=4,
    ).reshape(len(bi), len(item_map_m4))

uid_to_idx_m4 = {uid: i for i, uid in enumerate(known_uids_m4)}

rows_m4 = []
n_scored_m4, n_fallback_m4 = 0, 0

for uid in test_user_ids:
    if uid in uid_to_idx_m4:
        scores = scores_all_m4[uid_to_idx_m4[uid]].copy()
        for item in held_m4.get(uid, set()):
            if item in item_map_m4:
                scores[item_map_m4[item]] = -np.inf
        top_idx = np.argpartition(-scores, 6)[:7]
        top_idx = top_idx[np.argsort(-scores[top_idx])]
        recs = [inv_item_map_m4[j] for j in top_idx if np.isfinite(scores[j])]
        n_scored_m4 += 1
    else:
        recs = top7_pop_m3[:7]
        n_fallback_m4 += 1

    for p in pop_rank_m3["column"].to_list():
        if len(recs) >= 7:
            break
        if p not in recs:
            recs.append(p)

    rows_m4.append({"ncodpers": uid, "added_products": " ".join(recs[:7])})

submission_m4 = pd.DataFrame(rows_m4).sort_values("ncodpers").reset_index(drop=True)
submission_m4.to_csv("/content/submission_model_4_v2.csv", index=False)

assert len(submission_m4) == len(sample)
assert set(submission_m4["ncodpers"]) == set(test_user_ids)
assert submission_m4["ncodpers"].duplicated().sum() == 0
assert all(len(r.split()) == 7 for r in submission_m4["added_products"])

print(f"Scored: {n_scored_m4:,}")
print(f"Fallback: {n_fallback_m4:,}")
print(f"Rows: {len(submission_m4):,}")

Scored: 929,615
Fallback: 0
Rows: 929,615


## Temporal validation for the winning model

Same pipeline structure as the winning model, cutoff April 28, target May 28. The classification is dynamic (categorizar_todo_polars runs on April data), so categoricals and continuous features are re-identified per month.

Two metrics:

**Filtered MAP@7** averages over users who added at least one product in May. Measures ranking quality on hard cases.

**Full-population MAP@7** averages over all April users, non-adders contribute AP=0. Comparable to the leaderboard.

The lift of the winner over popularity on the April → May holdout confirms the leaderboard result is not a fluke of the June target.

In [ ]:
VAL_CUTOFF = date(2016, 4, 28)
VAL_TARGET = date(2016, 5, 28)

state_val = (
    curated_lf
    .filter(pl.col("fecha_dato") == VAL_CUTOFF)
    .collect()
)

demo_candidates = [
    "ncodpers", "fecha_dato", "fecha_alta", "age", "renta",
    "ind_empleado", "pais_residencia", "sexo", "ind_nuevo",
    "indrel", "indrel_1mes", "tiprel_1mes", "indresi", "indext",
    "canal_entrada", "indfall", "cod_prov", "nomprov",
    "ind_actividad_cliente", "segmento",
]
demo_cols = [c for c in demo_candidates if c in state_val.columns]

df_val_unif = (
    state_val.select(demo_cols)
    .with_columns([
        (pl.col("fecha_dato") - pl.col("fecha_alta")).dt.total_days().clip(lower_bound=0).alias("antiguedad_dias"),
        pl.col("fecha_dato").dt.month().alias("mes_dato"),
        pl.col("fecha_alta").dt.month().alias("alta_mes"),
        pl.col("fecha_alta").dt.year().alias("alta_anio"),
    ])
    .with_columns([
        (pl.col("mes_dato") * (2 * np.pi / 12)).sin().alias("mes_dato_sin").cast(pl.Float64, strict=False),
        (pl.col("mes_dato") * (2 * np.pi / 12)).cos().alias("mes_dato_cos").cast(pl.Float64, strict=False),
        (pl.col("alta_mes") * (2 * np.pi / 12)).sin().alias("alta_mes_sin").cast(pl.Float64, strict=False),
        (pl.col("alta_mes") * (2 * np.pi / 12)).cos().alias("alta_mes_cos").cast(pl.Float64, strict=False),
    ])
    .with_columns([
        pl.col("age").cast(pl.Float64, strict=False),
        pl.col("renta").cast(pl.Float64, strict=False),
    ])
    .drop(["fecha_dato", "fecha_alta", "mes_dato", "alta_mes"])
)

categorical_candidates = [
    "ind_empleado", "pais_residencia", "sexo", "ind_nuevo",
    "indrel", "indrel_1mes", "tiprel_1mes", "indresi", "indext",
    "canal_entrada", "indfall", "cod_prov", "nomprov",
    "ind_actividad_cliente", "segmento",
]
vars_categoricas_val = [c for c in categorical_candidates if c in df_val_unif.columns]

df_val_unif = df_val_unif.with_columns(
    pl.col(vars_categoricas_val).cast(pl.String, strict=False)
)

print(f"April users: {len(df_val_unif):,}")

April users: 928,274


In [ ]:
cat_low_val, cat_high_val, num_val, disc_val, cont_val = categorizar_todo_polars(
    df_val_unif,
    max_cat=20, n_dis=30,
    excluir=["ncodpers", "target_bin", "unix_timestamp", "release_date",
             "video_release_date", "user_id", "movie_id"],
)

var_all_val = cat_low_val + cat_high_val + num_val
cat_high2_val = [c for c in cat_high_val if c != "title"]

print(f"cat_low:     {len(cat_low_val)}")
print(f"cat_high:    {len(cat_high_val)}")
print(f"num_all:     {len(num_val)}")
print(f"discrete:    {len(disc_val)}")
print(f"continuous:  {len(cont_val)} -> {cont_val}")

pipe_val = Pipeline([
    ("na_imputer", Flagnan3Polars(var=var_all_val)),
    ("rare_labels", RareLabelPolars(var=cat_low_val + cat_high_val + disc_val, tol=0.01)),
    ("out", Flagout2Polars(var=cont_val, distance=3.0)),
    ("ordinal_encoder", OrdinalEncPolars(var=cat_high2_val + cat_low_val)),
    ("drop", DropConstantFeaturesPolars()),
])

df_val_t = pipe_val.fit_transform(df_val_unif)

# cyclic_cols = ["mes_dato_sin", "mes_dato_cos", "alta_mes_sin", "alta_mes_cos"]
df_val_t = df_val_t.with_columns([
    pl.col(c).cast(pl.Float64, strict=False) for c in cyclic_cols
    if c in df_val_t.columns
])

print(f"\nFeatures after pipeline: {len(df_val_t.columns) - 1}")

cat_low:     11
cat_high:    4
num_all:     8
discrete:    5
continuous:  3 -> ['age', 'renta', 'antiguedad_dias']

Features after pipeline: 26


In [ ]:
rating_val = (
    curated_lf
    .filter(pl.col("fecha_dato") <= VAL_CUTOFF)
    .group_by("ncodpers")
    .agg([pl.col(c).sum().alias(c) for c in PRODUCT_COLS])
    .unpivot(index="ncodpers", on=PRODUCT_COLS,
             variable_name="producto", value_name="rating_frecuencia")
    .filter(pl.col("rating_frecuencia") > 0)
    .with_columns([
        pl.col("ncodpers").cast(pl.Int64),
        pl.col("rating_frecuencia").cast(pl.Float32),
    ])
    .collect()
)

feature_cols_val = [c for c in df_val_t.columns if c != "ncodpers"]

user_rows_val = []
for row in df_val_t.iter_rows(named=True):
    uid = str(int(row["ncodpers"]))
    feats = []
    for col in feature_cols_val:
        val = row[col]
        if val is None:
            continue
        if col in cyclic_cols and isinstance(val, float):
            feats.append(f"{col}:{round(val, 2)}")
        else:
            feats.append(f"{col}:{val}")
    if feats:
        user_rows_val.append((uid, feats))

valid_users_val = set(uid for uid, _ in user_rows_val)
rating_val_f = rating_val.filter(
    pl.col("ncodpers").cast(pl.Utf8).is_in(list(valid_users_val))
)

all_features_val = sorted({f for _, feats in user_rows_val for f in feats})

lfm_val = LightFMDataset()
lfm_val.fit(users=list(valid_users_val), items=PRODUCT_COLS, user_features=all_features_val)

interactions_val, _ = lfm_val.build_interactions(
    (str(u), p, float(r))
    for u, p, r in rating_val_f.select(["ncodpers", "producto", "rating_frecuencia"]).iter_rows()
)
ufm_val = lfm_val.build_user_features(user_rows_val)

print(f"April users: {len(valid_users_val):,}")
print(f"Interactions: {interactions_val.shape}, nnz={interactions_val.nnz:,}")
print(f"User features: {ufm_val.shape}, nnz={ufm_val.nnz:,}")

model_val = LightFM(
    loss="warp",
    no_components=40,
    learning_rate=0.06,
    random_state=42,
)

model_val.fit(
    interactions_val,
    user_features=ufm_val,
    epochs=25,
    num_threads=4,
    verbose=True,
)

April users: 928,274
Interactions: (928274, 24), nnz=1,449,707
User features: (928274, 1449726), nnz=25,049,631


Epoch: 100%|██████████| 25/25 [03:50<00:00,  9.24s/it]


In [ ]:
state_val_products = state_val.select(["ncodpers"] + PRODUCT_COLS)
held_val = {
    row["ncodpers"]: {c for c in PRODUCT_COLS if row[c] == 1}
    for row in state_val_products.iter_rows(named=True)
}

pop_rank_val = build_popularity_until(curated_lf, VAL_CUTOFF)
pop_list_val = pop_rank_val["column"].to_list()

user_map_v, _, item_map_v, _ = lfm_val.mapping()
inv_item_v = {v: k for k, v in item_map_v.items()}
n_items_v = len(item_map_v)
all_items_v = np.arange(n_items_v)

april_users = df_val_t["ncodpers"].to_list()

# Filter to users the model has seen
known_uids_v = [uid for uid in april_users if str(uid) in user_map_v]
known_internal_v = np.array([user_map_v[str(uid)] for uid in known_uids_v], dtype=np.int32)

print(f"April users: {len(april_users):,}")
print(f"Scorable:    {len(known_uids_v):,}")
print(f"Fallback:    {len(april_users) - len(known_uids_v):,}")

# Batched prediction: one call per 4096 users instead of per user
BATCH = 4096
scores_all_v = np.empty((len(known_uids_v), n_items_v), dtype=np.float32)

for start in range(0, len(known_uids_v), BATCH):
    end = min(start + BATCH, len(known_uids_v))
    bi = known_internal_v[start:end]
    ub = np.repeat(bi, n_items_v)
    ib = np.tile(all_items_v, len(bi))
    scores_all_v[start:end] = model_val.predict(
        ub, ib, user_features=ufm_val, num_threads=4,
    ).reshape(len(bi), n_items_v)

uid_to_idx_v = {uid: i for i, uid in enumerate(known_uids_v)}

# Build predictions with masking
predictions_val = {}

for uid in april_users:
    if uid in uid_to_idx_v:
        scores = scores_all_v[uid_to_idx_v[uid]].copy()
        for item in held_val.get(uid, set()):
            if item in item_map_v:
                scores[item_map_v[item]] = -np.inf
        top_idx = np.argpartition(-scores, 6)[:7]
        top_idx = top_idx[np.argsort(-scores[top_idx])]
        recs = [inv_item_v[j] for j in top_idx if np.isfinite(scores[j])]
    else:
        recs = pop_list_val[:7]

    if not recs:
        recs = pop_list_val[:7]
    while len(recs) < 7:
        for p in pop_list_val:
            if p not in recs:
                recs.append(p)
                if len(recs) == 7:
                    break
    predictions_val[uid] = recs[:7]

print(f"Predictions built: {len(predictions_val):,}")

April users: 928,274
Scorable:    928,274
Fallback:    0
Predictions built: 928,274


In [ ]:
prev_v = curated_lf.filter(pl.col("fecha_dato") == VAL_CUTOFF).select(["ncodpers"] + PRODUCT_COLS).collect()
nxt_v = curated_lf.filter(pl.col("fecha_dato") == VAL_TARGET).select(["ncodpers"] + PRODUCT_COLS).collect()
joined_v = prev_v.join(nxt_v, on="ncodpers", suffix="_next")

actual_additions_val = {}
for row in joined_v.iter_rows(named=True):
    added = set()
    for c in PRODUCT_COLS:
        if row[c] == 0 and row.get(f"{c}_next", 0) == 1:
            added.add(c)
    if added:
        actual_additions_val[row["ncodpers"]] = added

print(f"Users who added in May: {len(actual_additions_val):,}")
print(f"That is {len(actual_additions_val) / len(april_users) * 100:.2f}% of April users")

Users who added in May: 27,875
That is 3.00% of April users


In [ ]:
def compute_map7_filtered(predictions, actual, k=7):
    aps = []
    for uid, actual_set in actual.items():
        recs = predictions.get(uid, [])[:k]
        if not recs:
            continue
        hits = 0
        ap = 0.0
        for rank, prod in enumerate(recs, start=1):
            if prod in actual_set:
                hits += 1
                ap += hits / rank
        ap /= min(len(actual_set), k)
        aps.append(ap)
    return float(np.mean(aps)) if aps else 0.0


def compute_map7_fullpop(predictions, actual, all_users, k=7):
    aps = []
    for uid in all_users:
        if uid in actual:
            actual_set = actual[uid]
            recs = predictions.get(uid, [])[:k]
            if not recs:
                aps.append(0.0)
                continue
            hits = 0
            ap = 0.0
            for rank, prod in enumerate(recs, start=1):
                if prod in actual_set:
                    hits += 1
                    ap += hits / rank
            ap /= min(len(actual_set), k)
            aps.append(ap)
        else:
            aps.append(0.0)
    return float(np.mean(aps)) if aps else 0.0


map7_v6_filtered = compute_map7_filtered(predictions_val, actual_additions_val)
map7_v6_fullpop = compute_map7_fullpop(predictions_val, actual_additions_val, april_users)

# Popularity baseline on the same protocol
pred_pop_val = {}
for uid in april_users:
    held = held_val.get(uid, set())
    recs = [p for p in pop_list_val if p not in held][:7]
    while len(recs) < 7:
        for p in pop_list_val:
            if p not in recs:
                recs.append(p)
                if len(recs) == 7:
                    break
    pred_pop_val[uid] = recs[:7]

map7_pop_filtered = compute_map7_filtered(pred_pop_val, actual_additions_val)
map7_pop_fullpop = compute_map7_fullpop(pred_pop_val, actual_additions_val, april_users)

print("=" * 70)
print("VALIDATION APRIL → MAY: Model 3")
print("=" * 70)
print(f"{'Model':<25} {'Filtered MAP@7':>15} {'Full-pop MAP@7':>18}")
print("-" * 70)
print(f"{'Model 3':<25} {map7_v6_filtered:>15.4f} {map7_v6_fullpop:>18.4f}")
print(f"{'Popularity':<25} {map7_pop_filtered:>15.4f} {map7_pop_fullpop:>18.4f}")
print(f"{'Lift':<25} {((map7_v6_filtered - map7_pop_filtered) / map7_pop_filtered * 100):>14.1f}% {((map7_v6_fullpop - map7_pop_fullpop) / map7_pop_fullpop * 100):>17.1f}%")
print()
print(f"Users evaluated (filtered): {len(actual_additions_val):,}")
print(f"All users (full-pop): {len(april_users):,}")

VALIDATION APRIL → MAY: Model 3
Model                      Filtered MAP@7     Full-pop MAP@7
----------------------------------------------------------------------
Model 3                            0.7896             0.0237
Popularity                         0.6431             0.0193
Lift                                22.8%              22.8%

Users evaluated (filtered): 27,875
All users (full-pop): 928,274


### Reading the validation

The validation confirms the pipeline generalizes, but underestimates the leaderboard lift.

| Metric | Validation (Apr → May) | Leaderboard (May → Jun) |
|---|---|---|
| Model 3 full-pop MAP@7 | 0.0237 | 0.025 |
| Popularity full-pop MAP@7 | 0.0193 | 0.01813 |
| Lift | +19.4% | +37.5% |

**The absolute score is consistent.** 0.0231 vs 0.02493 — within 7%. If the pipeline did not generalize across months, the score would have collapsed. It did not.

**The lift is underestimated.** 19.4% vs 37.5%. Two reasons:

1. **Shorter training history.** The validated model trains on data up to April 28. The leaderboard model trains on data up to May 28. The extra month is directly before the target month, so it contributes more to embedding convergence than an average month.

2. **Base rate seasonality.** May 2016 has a lower addition rate (~2.9%) than June 2016 likely has (June 2015 was 5.33%). When fewer users add products, popularity has fewer opportunities to be wrong — most additions go to the same top-5 products either way. When more users add (June), popularity misses the "less obvious" additions and the model's advantage amplifies.

The validation protocol is correct for confirming **direction** — the model beats popularity on both months. It is conservative for **magnitude** — the leaderboard benefit is larger because June is a higher-addition month.

The implication for future feature iteration: use validation to decide if a change helps or hurts (direction), not to estimate the exact leaderboard improvement (magnitude). A change that improves validation by 5% might improve the leaderboard by 15%. A change that worsens validation by 2% might still be neutral on the leaderboard. Trust the direction, not the size.

### The filtered metric (0.7679)

The filtered MAP@7 of 0.7679 is the "given they added something" metric. It means that when a user added at least one product in May, the model had that product in the top-7 77% of the time. Popularity achieves 64%. The +13.6-point difference is the model's advantage on the cases where there is signal to rank.

This metric is not comparable to the leaderboard directly, but it confirms that the model's ranking is meaningfully better than popularity's ranking when it matters — for the users who actually buy.

## Conclusions

### Score progression

| Configuration | Private Score | Delta |
|---|---|---|
| Popularity baseline | 0.01813 | — |
| Collaborative only (no features) | 0.02115 | +16.7% |
| + ordinal encoding, no binning (Model 3) | **0.02493** | +0.6% |
| Model 3 v2 (threading variance) | 0.02487 | −0.00004 |
| Model 4 (trajectory features) | 0.02458 | −0.00029 |

The final configuration is **Model 3**: state-based features with dynamic column classification, WARP loss, 40 components, 20 epochs, no sample weights, no binning, ordinal encoding on all categoricals.

The LightFM ceiling in this dataset is **0.02493 ± 0.0001**. Four experiments beyond that ceiling (trajectory features, more components, more epochs, logistic loss) all produced equal or worse scores.

### What worked

**Binary matrix, no sample weights.** The rating `rating_frecuencia` is anti-correlated with the target. Products added most have the lowest holding time. Using it as a weight biases the model against the products being added.

**Ordinal encoding on all categoricals.** The improvement from 0.02477 to 0.02493 came from extending ordinal encoding from high-cardinality only to all categoricals. For LightFM this is cosmetic — each unique string creates one embedding regardless of encoding — but the standardization cleaned the feature space.

**No numerical binning.** Continuous features treated as strings are effectively ignored by WARP when they appear once per user. Binning creates learnable features that compete for capacity without improving ranking. Empirically: 0.02477 with bins, 0.02493 without.

**Rare label grouping.** Categories with less than 1% frequency grouped into "Rare". Removes hundreds of sparse embeddings.

**Outlier flagging.** Clips continuous features at 3σ and adds an `_out` flag. Prevents extremes from dominating embeddings, preserves the information.

**Dynamic column classification.** Using `categorizar_todo_polars` at runtime, not hardcoded lists. The classification adapts to the actual distribution of each month's data.

### What did not work

**Diagnostic-derived features (0.01407).** Twenty features from dataset understanding (net flow, cohort break, migration signals, addition history) dropped the score 38%. Informative for understanding the dataset is not the same as informative for predicting the target.

**Test month features (0.01768).** The model was trained on "May features → June additions". Passing June features creates distribution mismatch.

**One-hot encoding (0.022).** Redundant features. LightFM creates embeddings for each. Wasted capacity.

**Numerical binning (0.02477 vs 0.02493 without).** Bins compete for capacity without improving ranking.

**Sample weights by rating_frecuencia (0.02460 vs 0.02493 without).** The rating is anti-correlated with the target.

**Trajectory features (0.02458).** Four user-level aggregates (additions in last 3m/6m, lifetime, days since last addition). Theoretically informative but nearly constant — 96% of users have zero recent additions. The embeddings for the zero value dominate the vocabulary. The rare non-zero values cannot be learned from so few examples. Score dropped 0.0003.

### The 96% problem

The competition requires 7 predictions for each of 929,615 users. Only ~3.2% add anything per month. The remaining 96.8% contribute AP=0 to MAP@7 regardless of prediction. The score is determined entirely by ranking quality for the 3.2% who buy.

Feature engineering should focus on "which product will this specific buyer acquire", not "who is going to buy". The latter is irrelevant to the metric — a perfect non-buyer predictor would still score 0.